# LeetCode 269 — Alien Dictionary (Hard, Premium): nhập môn đồ thị có hướng & Topological Sort bằng Python

> **Ghi chú Premium**: đây là bài **Premium** (bị khóa với tài khoản LeetCode miễn phí). Phát biểu đề, ví dụ, constraints và signature dưới đây được viết lại theo bản LeetCode hiện hành dựa trên hiểu biết của mình, không tự thêm ràng buộc nào. Không có Premium bạn vẫn học trọn vẹn bài này ngay trong notebook: mọi lời giải đều chạy và **tự kiểm tra** tại chỗ.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def alienOrder(self, words: List[str]) -> str:

```

Câu chuyện: một ngôn ngữ ngoài hành tinh dùng đúng 26 chữ cái tiếng Anh nhưng **thứ tự chữ cái khác hẳn**. Ta chỉ được xem vài trang "từ điển" của họ (một danh sách từ đã sắp xếp theo luật của họ) và phải **đoán lại bảng chữ cái**. Đằng sau câu chuyện là một kỹ thuật kinh điển, gặp rất nhiều trong phỏng vấn: **sắp xếp tô-pô (topological sort)** trên **đồ thị có hướng**.

---

## Sau notebook này, bạn sẽ nắm được

1. **Thứ tự từ điển** (lexicographic order): luật "chữ khác nhau đầu tiên quyết định" và luật "tiền tố đứng trước".
2. Kiểm tra một danh sách từ có được sắp xếp theo một bảng chữ cái **cho trước** hay không (chính là bài #953) — công cụ "trọng tài" dùng suốt notebook.
3. **Cách 1 — Brute force**: thử mọi hoán vị chữ cái bằng `itertools.permutations`, và thấy vì sao `k!` bùng nổ.
4. **Bước đột phá**: so sánh hai từ liền kề → rút ra **một luật "chữ A đứng trước chữ B"** (một **cạnh**), và nhận ra trường hợp **tiền tố không hợp lệ** (`"abc"` đứng trước `"ab"`).
5. **Đồ thị có hướng** từ con số 0: danh sách kề (adjacency list), **bậc vào** (in-degree), **chu trình** (cycle), **topological order**.
6. **Cách 2 — so sánh MỌI cặp từ** rồi chọn dần chữ "tự do"; hiểu thật kỹ **vì sao chỉ cần so sánh các từ liền kề**.
7. **Cách 3 (chính) — thuật toán Kahn**: BFS theo in-degree với `collections.deque`, có trace hàng đợi từng bước.
8. **Cách 4 — DFS 3 màu** (trắng / xám / đen) để phát hiện chu trình, rồi đảo ngược thứ tự **post-order**.
9. Viết `class Solution` đúng format LeetCode và **test bằng hàm kiểm tra tính hợp lệ** (vì đáp án không duy nhất), đối chiếu chéo ngẫu nhiên với brute force.
10. Lỗi hay gặp, mẹo nhận dạng, biến thể (#207, #210, #953, #444...) và 6 bài tập có đáp án ở cuối.

Lộ trình: `Thứ tự từ điển` → `Brute force hoán vị` → `Rút cạnh từ cặp từ` → `Đồ thị + chu trình` → `So sánh mọi cặp` → `Kahn (BFS)` → `DFS 3 màu`.

## 1. Đề bài

Có một ngôn ngữ ngoài hành tinh mới dùng **bảng chữ cái tiếng Anh**. Tuy nhiên, **thứ tự giữa các chữ cái** thì bạn không biết.

Bạn được cho một danh sách chuỗi `words` lấy từ từ điển của ngôn ngữ đó. Người ta khẳng định rằng các chuỗi trong `words` **đã được sắp xếp theo thứ tự từ điển** (sorted lexicographically) theo luật của ngôn ngữ mới.

- Nếu khẳng định này **sai**, tức là cách sắp xếp của `words` **không thể** ứng với bất kỳ thứ tự chữ cái nào, hãy trả về `""`.
- Ngược lại, trả về một chuỗi gồm **các chữ cái phân biệt** của ngôn ngữ mới, sắp theo thứ tự tăng dần theo luật của ngôn ngữ mới. Nếu có **nhiều đáp án**, trả về **bất kỳ** đáp án nào.

Định nghĩa trong đề: chuỗi `s` **nhỏ hơn theo thứ tự từ điển** chuỗi `t` nếu tại **vị trí đầu tiên** hai chuỗi khác nhau, chữ của `s` đứng trước chữ của `t` trong ngôn ngữ ngoài hành tinh. Nếu `min(s.length, t.length)` chữ đầu tiên giống hệt nhau thì `s` nhỏ hơn **khi và chỉ khi** `s.length < t.length`.

**Example 1**

```text
Input: words = ["wrt","wrf","er","ett","rftt"]
Output: "wertf"
```

**Example 2**

```text
Input: words = ["z","x"]
Output: "zx"
```

**Example 3**

```text
Input: words = ["z","x","z"]
Output: ""
Explanation: The order is invalid, so return "".
```

**Constraints**

- `1 <= words.length <= 100`
- `1 <= words[i].length <= 100`
- `words[i]` chỉ gồm chữ cái tiếng Anh viết thường.

Ghi chú quan trọng:

- Đáp án phải chứa **đúng các chữ cái xuất hiện trong `words`**, mỗi chữ **đúng một lần**: không thiếu chữ nào, không thêm chữ không xuất hiện.
- Vì "nhiều đáp án thì trả về bất kỳ", ta **không** test bằng cách so với một chuỗi cố định; ta sẽ viết một **hàm kiểm tra tính hợp lệ** của thứ tự (mục 13).

Ký hiệu dùng trong notebook:

- `n = len(words)` (≤ 100), `L` = độ dài từ dài nhất (≤ 100);
- `C` = tổng số ký tự của mọi từ (≤ 100 × 100 = 10^4);
- `k` = số chữ cái **phân biệt** xuất hiện (≤ 26);
- `E` = số "luật" (cạnh) phân biệt rút ra được.

In [1]:
from typing import List
from collections import deque
from itertools import permutations
import math
import random

ex1 = ["wrt", "wrf", "er", "ett", "rftt"]   # Example 1, mong đợi "wertf"
ex2 = ["z", "x"]                            # Example 2, mong đợi "zx"
ex3 = ["z", "x", "z"]                       # Example 3, mong đợi ""

for name, words in [("ex1", ex1), ("ex2", ex2), ("ex3", ex3)]:
    print(f"{name}: n = {len(words)}, words = {words}")

# set(...) = tập hợp, tự bỏ chữ trùng; sorted(...) để in ra theo thứ tự cố định
print("Các chữ cái xuất hiện trong ex1:", sorted(set("".join(ex1))))

ex1: n = 5, words = ['wrt', 'wrf', 'er', 'ett', 'rftt']
ex2: n = 2, words = ['z', 'x']
ex3: n = 3, words = ['z', 'x', 'z']
Các chữ cái xuất hiện trong ex1: ['e', 'f', 'r', 't', 'w']


### Đọc đề bằng trực giác

Hãy tưởng tượng bạn là nhà khảo cổ nhặt được 5 dòng đầu của một cuốn từ điển ngoài hành tinh: `wrt, wrf, er, ett, rftt`. Từ điển luôn xếp từ "nhỏ" trước, từ "lớn" sau, nên mỗi cặp từ đứng cạnh nhau **lộ ra một chút thông tin** về bảng chữ cái:

- `"wrt"` đứng trước `"wrf"`: giống nhau ở `w`, `r`, khác nhau lần đầu ở vị trí 2: `t` vs `f` → trong bảng chữ cái của họ, **`t` đứng trước `f`**.
- `"wrf"` đứng trước `"er"`: khác nhau ngay vị trí 0: `w` vs `e` → **`w` đứng trước `e`**.
- ... (làm đầy đủ ở mục 5)

Ghép các mẩu thông tin lại ta được `w → e → r → t → f`, tức đáp án `"wertf"`.

Example 3 `["z","x","z"]`: cặp đầu nói `z` trước `x`, cặp sau nói `x` trước `z`. Hai điều này **mâu thuẫn** (không thể vừa `z < x` vừa `x < z`) → không có bảng chữ cái nào thỏa mãn → trả `""`.

Còn một kiểu "không hợp lệ" nữa, tinh vi hơn: `["abc", "ab"]`. Theo luật từ điển, `"ab"` là **tiền tố** (prefix — phần đầu) của `"abc"` nên `"ab"` luôn phải đứng **trước** `"abc"`, bất kể bảng chữ cái là gì. Danh sách lại đặt `"abc"` trước → sai trong **mọi** bảng chữ cái → `""`.

Tóm lại có **hai nguồn gốc** của đáp án `""`: **mâu thuẫn vòng tròn** (sau này gọi là chu trình) và **từ dài đứng trước tiền tố của nó**. Notebook sẽ dạy cách phát hiện cả hai.

## 2. Nền tảng 1 — Thứ tự từ điển (lexicographic order)

So sánh hai từ `s` và `t` giống hệt cách bạn tra từ điển tiếng Anh:

**Luật 1 — chữ khác nhau đầu tiên quyết định.** Đi từ trái sang phải, tìm vị trí `i` **đầu tiên** mà `s[i] != t[i]`. Từ nào có chữ đứng trước (trong bảng chữ cái) ở vị trí đó thì đứng trước. Các chữ **phía sau** vị trí `i` **không còn quan trọng nữa**.

**Luật 2 — tiền tố đứng trước.** Nếu phần chung (độ dài bằng từ ngắn hơn) giống hệt nhau, tức là từ ngắn là **tiền tố** của từ dài, thì từ **ngắn hơn** đứng trước. Hai từ giống hệt nhau thì "bằng nhau" — đứng cạnh nhau thế nào cũng được.

Ví dụ với bảng chữ cái tiếng Anh bình thường `a < b < c < ... < z`:

| `s` | `t` | Vị trí khác đầu tiên | Kết luận | Luật |
|---|---|---|---|---|
| `"apple"` | `"apply"` | 4: `e` vs `y` | `"apple" < "apply"` vì `e < y` | 1 |
| `"cat"` | `"car"` | 2: `t` vs `r` | `"car" < "cat"` vì `r < t` | 1 |
| `"b"` | `"abc"` | 0: `b` vs `a` | `"abc" < "b"` (dù `"abc"` dài hơn!) | 1 |
| `"app"` | `"apple"` | không có trong phần chung | `"app" < "apple"` vì ngắn hơn | 2 |
| `"cat"` | `"cat"` | không có | bằng nhau | 2 |

Chú ý dòng `"b"` vs `"abc"`: độ dài **chỉ** được xét khi phần chung giống hệt nhau. Python so sánh chuỗi bằng `<`, `>`, `==` theo đúng hai luật này, với thứ tự chữ cái tiếng Anh bình thường (chính xác là theo mã Unicode của ký tự). Ta thử ngay:

In [2]:
print("apple" < "apply")   # True : vị trí 4, e < y
print("car" < "cat")       # True : vị trí 2, r < t
print("abc" < "b")         # True : vị trí 0, a < b — độ dài không quan trọng
print("app" < "apple")     # True : "app" là tiền tố của "apple"
print("cat" == "cat")      # True : bằng nhau
print(sorted(["cat", "b", "apple", "app", "car", "abc"]))

True
True
True
True
True
['abc', 'app', 'apple', 'b', 'car', 'cat']


### Hai công cụ Python sẽ dùng nhiều: `zip` và `for ... else`

**`zip(a, b)`** ghép từng phần tử của `a` với phần tử **cùng vị trí** của `b` thành từng cặp, và **dừng ở dãy ngắn hơn**:

```python
list(zip("wrt", "wrf"))  # [('w','w'), ('r','r'), ('t','f')]
list(zip("abc", "ab"))   # [('a','a'), ('b','b')]  — 'c' bị bỏ vì "ab" đã hết
```

Đây đúng là thứ ta cần để duyệt "phần chung" của hai từ. Mẹo thứ hai: `zip(words, words[1:])` cho ra **các cặp từ liền kề** `(words[0], words[1])`, `(words[1], words[2])`, ... vì `words[1:]` là chính danh sách đó nhưng **lệch đi một vị trí**.

**`for ... else`** là cú pháp hơi lạ của Python: khối `else` gắn với vòng `for` chạy **khi vòng lặp kết thúc bình thường, KHÔNG gặp `break`**. Hãy đọc nó là "nếu duyệt hết mà không `break` thì...":

```python
for a, b in zip(w1, w2):
    if a != b:
        ...        # tìm thấy chữ khác nhau đầu tiên
        break
else:
    ...            # duyệt hết phần chung mà không thấy chữ nào khác -> trường hợp tiền tố
```

Mẫu này chính là xương sống của bước "so sánh hai từ" trong lời giải LeetCode cuối cùng.

In [3]:
print(list(zip("wrt", "wrf")))
print(list(zip("abc", "ab")))           # dừng ở chuỗi ngắn hơn

words = ["wrt", "wrf", "er"]
print(list(zip(words, words[1:])))     # các cặp liền kề

def first_diff_demo(w1, w2):
    for a, b in zip(w1, w2):
        if a != b:
            print(f"{w1!r} vs {w2!r}: khác nhau ở {a!r}/{b!r} -> break, khối else KHÔNG chạy")
            break
    else:
        print(f"{w1!r} vs {w2!r}: không gặp break -> khối else chạy (phần chung giống hệt)")

first_diff_demo("wrt", "wrf")
first_diff_demo("abc", "ab")

[('w', 'w'), ('r', 'r'), ('t', 'f')]
[('a', 'a'), ('b', 'b')]
[('wrt', 'wrf'), ('wrf', 'er')]
'wrt' vs 'wrf': khác nhau ở 't'/'f' -> break, khối else KHÔNG chạy
'abc' vs 'ab': không gặp break -> khối else chạy (phần chung giống hệt)


## 3. Nền tảng 2 — So sánh từ theo một bảng chữ cái "lạ" cho trước

Giả sử ai đó đưa ta bảng chữ cái của người ngoài hành tinh, ví dụ `order = "wertf"`. Làm sao kiểm tra `"wrt"` có đứng trước `"er"` không, khi `<` của Python chỉ biết thứ tự `a, b, c, ...`?

**Bước 1 — đánh số thứ hạng (rank) cho mỗi chữ** bằng `dict`: chữ đứng đầu có rank 0, chữ tiếp theo rank 1, ...

```python
rank = {ch: i for i, ch in enumerate(order)}   # {'w': 0, 'e': 1, 'r': 2, 't': 3, 'f': 4}
```

`enumerate(order)` sinh các cặp `(0, 'w'), (1, 'e'), ...`; còn dict comprehension `{key: value for ...}` dựng một dict từ các cặp đó.

**Bước 2 — đổi mỗi từ thành list các rank**: `"wrt"` → `[0, 2, 3]`, `"er"` → `[1, 2]`.

**Bước 3 — so sánh hai list bằng `<=` có sẵn của Python.** Python so sánh list **cũng theo đúng 2 luật từ điển**: phần tử khác nhau đầu tiên quyết định; nếu list này là tiền tố của list kia thì list ngắn hơn nhỏ hơn. Ví dụ `[0, 2, 3] < [1, 2]` là `True` vì `0 < 1` ngay vị trí đầu. Mẹo này giúp ta **không phải tự viết** hàm so sánh từ: "dịch" chữ cái lạ thành số rồi để Python lo phần còn lại.

**Bước 4 — một danh sách là "đã sắp xếp" khi mọi cặp liền kề thỏa `key[i] <= key[i+1]`.** Dùng `<=` (không phải `<`) vì hai từ giống nhau được phép đứng cạnh nhau, ví dụ `["z", "z"]`. Hàm `all(...)` trả `True` khi **mọi** phần tử đều đúng (và dừng ngay khi gặp phần tử sai).

Bước 4 chính là bài LeetCode **#953 Verifying an Alien Dictionary** (Easy). Ta gọi nó là `is_sorted_by_order` và dùng làm **trọng tài** cho cả notebook. Lưu ý: hàm cần `order` chứa đủ mọi chữ xuất hiện trong `words` (nếu không, `rank[ch]` báo `KeyError`).

In [4]:
order = "wertf"
rank = {ch: i for i, ch in enumerate(order)}
print("rank =", rank)

def word_key(word, rank):
    return [rank[ch] for ch in word]      # "dịch" từ thành list số thứ hạng

for w in ex1:
    print(f"{w:>5} -> {word_key(w, rank)}")

print([0, 2, 3] < [1, 2])     # True : 0 < 1 ngay vị trí đầu
print([1, 2] < [1, 2, 0])     # True : list ngắn là tiền tố -> nhỏ hơn
print([3, 1] < [3, 0, 5])     # False: vị trí 1 có 1 > 0

rank = {'w': 0, 'e': 1, 'r': 2, 't': 3, 'f': 4}
  wrt -> [0, 2, 3]
  wrf -> [0, 2, 4]
   er -> [1, 2]
  ett -> [1, 3, 3]
 rftt -> [2, 4, 3, 3]
True
True
False


In [5]:
def is_sorted_by_order(words, order):
    # True nếu `words` được sắp xếp (không giảm) theo bảng chữ cái `order`
    rank = {ch: i for i, ch in enumerate(order)}
    keys = [[rank[ch] for ch in w] for w in words]
    return all(keys[i] <= keys[i + 1] for i in range(len(keys) - 1))

print(is_sorted_by_order(ex1, "wertf"))                       # True
print(is_sorted_by_order(ex1, "abcdefghijklmnopqrstuvwxyz"))  # False: "wrt" > "wrf" vì t > f
print(is_sorted_by_order(ex1, "wretf"))                       # False: "ett" > "rftt" vì e đứng sau r
print(is_sorted_by_order(["z", "z"], "z"))                    # True : trùng nhau vẫn hợp lệ
print(is_sorted_by_order(["abc", "ab"], "abc"))               # False: tiền tố đứng sau

True
False
False
True
False


## 4. Cách 1 — Brute force: thử mọi bảng chữ cái có thể

Cách người mới nghĩ ra đầu tiên: *"Mình đã có trọng tài `is_sorted_by_order`. Vậy cứ **thử hết** mọi cách xếp các chữ cái; cách nào làm `words` được sắp xếp thì trả về cách đó; thử hết mà không có thì trả `""`."*

Cần hai thứ:

**(a) Danh sách chữ cái phân biệt.** Ghép mọi từ thành một chuỗi rồi lấy mỗi chữ một lần. Ta dùng `dict.fromkeys(chuỗi)`: tạo dict có các key lấy từ chuỗi (key trùng tự gộp lại). Vì dict **giữ thứ tự chèn**, ta được các chữ theo **thứ tự xuất hiện đầu tiên** — ổn định và dễ đoán. (Không dùng `set` để lấy thứ tự, vì thứ tự duyệt của `set` chuỗi có thể **khác nhau giữa các lần chạy Python**.)

**(b) Mọi hoán vị.** Một hoán vị (permutation) của `k` chữ là một cách xếp `k` chữ đó thành hàng. `itertools.permutations(seq)` sinh lần lượt **mọi** hoán vị dưới dạng tuple. Với 3 chữ `a, b, c` có `3! = 3 · 2 · 1 = 6` hoán vị: chọn chữ đứng đầu (3 cách) × chữ thứ hai (2 cách còn lại) × chữ cuối (1 cách).

In [6]:
print("".join(ex1))                            # ghép mọi từ thành một chuỗi
letters = list(dict.fromkeys("".join(ex1)))
print("Chữ cái phân biệt theo thứ tự xuất hiện:", letters)

for p in permutations("abc"):
    print(p, "->", "".join(p))
print("Số hoán vị của 3 chữ:", len(list(permutations("abc"))))

wrtwrferettrftt
Chữ cái phân biệt theo thứ tự xuất hiện: ['w', 'r', 't', 'f', 'e']
('a', 'b', 'c') -> abc
('a', 'c', 'b') -> acb
('b', 'a', 'c') -> bac
('b', 'c', 'a') -> bca
('c', 'a', 'b') -> cab
('c', 'b', 'a') -> cba
Số hoán vị của 3 chữ: 6


### Code brute force

```text
letters = các chữ phân biệt
với mỗi hoán vị perm của letters:
    nếu is_sorted_by_order(words, perm): trả về "".join(perm)
trả về ""
```

Ta viết thêm `count_valid_orders(words)` đếm **tất cả** hoán vị hợp lệ. Hàm này giúp thấy tận mắt "đáp án không duy nhất", và về sau làm **trọng tài** cho các cách nhanh hơn trên input nhỏ.

`is_sorted_by_order` nhận `order` là chuỗi hay tuple đều được, vì `enumerate` chạy trên cả hai.

In [7]:
def unique_letters(words):
    return list(dict.fromkeys("".join(words)))     # chữ phân biệt, theo thứ tự xuất hiện

def alien_order_brute(words):
    for perm in permutations(unique_letters(words)):
        if is_sorted_by_order(words, perm):
            return "".join(perm)
    return ""

def count_valid_orders(words):
    return sum(1 for perm in permutations(unique_letters(words))
               if is_sorted_by_order(words, perm))

for words in [ex1, ex2, ex3, ["abc", "ab"], ["ab", "adc"], ["zy", "zx"]]:
    print(f"{str(words):36} brute = {alien_order_brute(words)!r:8} số thứ tự hợp lệ = {count_valid_orders(words)}")

['wrt', 'wrf', 'er', 'ett', 'rftt']  brute = 'wertf'  số thứ tự hợp lệ = 1
['z', 'x']                           brute = 'zx'     số thứ tự hợp lệ = 1
['z', 'x', 'z']                      brute = ''       số thứ tự hợp lệ = 0
['abc', 'ab']                        brute = ''       số thứ tự hợp lệ = 0
['ab', 'adc']                        brute = 'abdc'   số thứ tự hợp lệ = 12
['zy', 'zx']                         brute = 'zyx'    số thứ tự hợp lệ = 3


### Nút thắt của brute force

Output trên cho thấy hai điều thú vị:

1. `["ab", "adc"]` có tới **12** thứ tự hợp lệ: ta chỉ biết đúng một luật `b` trước `d` (vị trí 1); còn `a` và `c` có thể đứng **bất kỳ đâu**. Đó là lý do đề cho phép "trả về bất kỳ".
2. `["zy", "zx"]`: chỉ biết `y` trước `x`; `z` không bị ràng buộc gì (vì `z` chỉ nằm ở vị trí giống nhau của hai từ) → 3 thứ tự: `zyx`, `yzx`, `yxz`.

Nhưng chi phí thì sao? Với `k` chữ có `k!` hoán vị, mỗi hoán vị lại phải kiểm tra cả danh sách (`O(C)`). Tổng cộng `O(k! · C)`. Bảng dưới cho thấy `k!` tăng khủng khiếp; đề cho phép tới 26 chữ cái → `26! ≈ 4 · 10^26` hoán vị — máy chạy tới lúc vũ trụ già đi cũng chưa xong.

→ Brute force **chỉ dùng được khi `k` nhỏ (khoảng ≤ 8)**. Ta vẫn giữ nó làm **trọng tài** để kiểm chứng các cách nhanh hơn trên input nhỏ.

In [8]:
for k in [3, 5, 8, 10, 15, 26]:
    print(f"k = {k:2} chữ cái -> k! = {math.factorial(k):,}")

k =  3 chữ cái -> k! = 6
k =  5 chữ cái -> k! = 120
k =  8 chữ cái -> k! = 40,320
k = 10 chữ cái -> k! = 3,628,800
k = 15 chữ cái -> k! = 1,307,674,368,000
k = 26 chữ cái -> k! = 403,291,461,126,605,635,584,000,000


## 5. Bước đột phá — mỗi cặp từ liền kề "khai" ra tối đa một luật

Brute force thử mù quáng. Cách thông minh hơn: **đọc luật trực tiếp từ dữ liệu**. Theo Luật 1, khi `w1` đứng trước `w2` và chúng khác nhau lần đầu ở vị trí `i` với `w1[i] = a`, `w2[i] = b`, thì **bắt buộc** `a` đứng trước `b` trong bảng chữ cái. Ta ghi luật này là `a → b` (đọc: "a trước b").

So sánh hai từ `w1` (đứng trước) và `w2` (đứng sau) chỉ có **3 kết cục**:

| Kết cục | Ví dụ | Ta học được gì |
|---|---|---|
| **edge**: có vị trí khác nhau đầu tiên `a != b` | `"wrt"`, `"wrf"` → `t`, `f` | luật `t → f`. **Dừng ngay**: các chữ sau đó không nói gì cả |
| **ok**: `w1` là tiền tố của `w2` (hoặc hai từ bằng nhau) | `"ab"`, `"abc"` / `"z"`, `"z"` | không có luật nào, nhưng cũng không sai |
| **invalid**: `w2` là tiền tố **thật sự** của `w1` (`w1` dài hơn) | `"abc"`, `"ab"` | sai với **mọi** bảng chữ cái → cả bài trả `""` |

Tại sao **phải dừng** ngay sau chữ khác nhau đầu tiên? Xét `"ab"` đứng trước `"ba"`: vị trí 0 cho luật `a → b`. Vị trí 1 là `b` vs `a` — nếu (sai lầm) rút thêm `b → a` thì thành mâu thuẫn, trong khi thực tế `["ab", "ba"]` hoàn toàn hợp lệ với bảng chữ cái `"ab"`. Theo Luật 1, chữ phía sau vị trí khác nhau đầu tiên **không ảnh hưởng** tới thứ tự.

Áp dụng bằng tay cho Example 1:

| Cặp liền kề | Vị trí khác đầu tiên | Luật rút ra |
|---|---|---|
| `wrt` / `wrf` | 2: `t` vs `f` | `t → f` |
| `wrf` / `er` | 0: `w` vs `e` | `w → e` |
| `er` / `ett` | 1: `r` vs `t` | `r → t` |
| `ett` / `rftt` | 0: `e` vs `r` | `e → r` |

Nối lại: `w → e → r → t → f` → `"wertf"`. Đúng Output của đề! Hàm `compare_pair` dưới đây làm đúng việc trên.

In [9]:
def compare_pair(w1, w2):
    # So sánh w1 (đứng trước) với w2 (đứng sau). Trả về:
    #   ("edge", a, b)        nếu rút được luật a -> b
    #   ("ok", None, None)    nếu w1 là tiền tố của w2 (hoặc bằng nhau)
    #   ("invalid", None, None) nếu w2 là tiền tố thật sự của w1
    for a, b in zip(w1, w2):
        if a != b:
            return ("edge", a, b)          # chữ khác nhau đầu tiên -> dừng ngay
    if len(w1) > len(w2):
        return ("invalid", None, None)
    return ("ok", None, None)

for w1, w2 in zip(ex1, ex1[1:]):
    print(f"{w1:>5} / {w2:<5} -> {compare_pair(w1, w2)}")
print()
for w1, w2 in [("ab", "abc"), ("z", "z"), ("abc", "ab"), ("ab", "ba")]:
    print(f"{w1:>5} / {w2:<5} -> {compare_pair(w1, w2)}")

  wrt / wrf   -> ('edge', 't', 'f')
  wrf / er    -> ('edge', 'w', 'e')
   er / ett   -> ('edge', 'r', 't')
  ett / rftt  -> ('edge', 'e', 'r')

   ab / abc   -> ('ok', None, None)
    z / z     -> ('ok', None, None)
  abc / ab    -> ('invalid', None, None)
   ab / ba    -> ('edge', 'a', 'b')


## 6. Nền tảng 3 — Đồ thị có hướng (directed graph)

Các luật `a → b` tạo thành một **đồ thị có hướng**:

- **Đỉnh** (node / vertex): mỗi chữ cái xuất hiện.
- **Cạnh có hướng** (directed edge) `a → b`: "a phải đứng trước b". Cạnh có **chiều**: `a → b` khác `b → a`.

Cách lưu đồ thị phổ biến nhất trong Python là **danh sách kề** (adjacency list): một `dict` mà `adj[a]` là list các đỉnh mà `a` chỉ tới. Example 1:

```python
adj = {"w": ["e"], "e": ["r"], "r": ["t"], "t": ["f"], "f": []}
```

**Bậc vào** (in-degree) của đỉnh `v` = số cạnh **đi vào** `v` = số chữ **bắt buộc phải đứng ngay trước** `v` theo các luật trực tiếp. Đỉnh có in-degree 0 là chữ "không ai bắt phải đứng sau" → có thể đứng đầu.

Một ví dụ đời thường để hình dung (sẽ dùng lại khi học thuật toán Kahn): mặc đồ buổi sáng.

```text
  quần ──► giày ◄── tất
    │
    ▼
 thắt lưng ◄── áo
```

`quần → giày` (mặc quần trước rồi mới xỏ giày), `tất → giày`, `quần → thắt lưng`, `áo → thắt lưng`. In-degree: `giày` = 2, `thắt lưng` = 2, còn `quần`, `tất`, `áo` = 0.

Hai chi tiết cài đặt quan trọng:

1. **Mọi đỉnh phải có mặt** trong `adj`, kể cả đỉnh không có cạnh nào (list rỗng `[]`). Nếu quên, chữ đó sẽ "biến mất" khỏi đáp án.
2. Ta dùng **list** cho `adj[a]` và kiểm tra `if b not in adj[a]` để **không thêm trùng** cạnh. Dùng `set` cũng được (tra cứu nhanh hơn), nhưng thứ tự duyệt của `set` chuỗi có thể **đổi giữa các lần chạy Python** (do hash ngẫu nhiên), làm trace khó đối chiếu; với tối đa 26 chữ, list là đủ nhanh.

In [10]:
dress = {
    "quần": ["giày", "thắt lưng"],
    "tất": ["giày"],
    "áo": ["thắt lưng"],
    "giày": [],
    "thắt lưng": [],
}

def compute_indegree(adj):
    indeg = {u: 0 for u in adj}          # mọi đỉnh bắt đầu từ 0
    for u in adj:
        for v in adj[u]:
            indeg[v] += 1                # cạnh u -> v làm in-degree của v tăng 1
    return indeg

print("Danh sách kề:")
for u, vs in dress.items():
    print(f"  {u:9} -> {vs}")
print("In-degree:", compute_indegree(dress))

Danh sách kề:
  quần      -> ['giày', 'thắt lưng']
  tất       -> ['giày']
  áo        -> ['thắt lưng']
  giày      -> []
  thắt lưng -> []
In-degree: {'quần': 0, 'tất': 0, 'áo': 0, 'giày': 2, 'thắt lưng': 2}


### Chu trình (cycle) — vì sao nó nghĩa là "không hợp lệ"

**Chu trình** là một đường đi theo chiều mũi tên quay về chính đỉnh xuất phát, ví dụ `z → x → z` (Example 3), hoặc dài hơn `a → b → c → a`:

```text
Example 3:  z ──► x        a ──► b
            ▲     │        ▲     │
            └─────┘        └─ c ◄┘
```

Nếu có chu trình `a → b → c → a`, bảng chữ cái phải có `a` trước `b`, `b` trước `c`, `c` trước `a` → suy ra `a` đứng trước chính nó: vô lý. Vậy **có chu trình ⇒ không có đáp án**. Ngược lại (sẽ chứng minh ở mục 14), nếu **không có chu trình** thì **luôn** xếp được một thứ tự thỏa mọi cạnh.

### Gom lại: hàm `build_graph(words)`

1. Tạo đỉnh cho **mọi** chữ xuất hiện (theo thứ tự xuất hiện đầu tiên): `adj[ch] = []`, `indeg[ch] = 0`.
2. Với từng cặp **liền kề** `(w1, w2)`, gọi `compare_pair`:
   - `"invalid"` → trả `None` (báo hiệu "tiền tố đứng sau, cả bài là `""`");
   - `"edge"` với luật `a → b` **chưa có** → thêm `b` vào `adj[a]`, tăng `indeg[b]`;
   - `"ok"` → bỏ qua.

Tại sao chỉ xét cặp **liền kề** là đủ? Câu hỏi rất hay — ta sẽ trả lời cẩn thận ở mục 7.

In [11]:
def build_graph(words):
    # Trả về (adj, indeg) hoặc None nếu gặp trường hợp tiền tố không hợp lệ
    adj, indeg = {}, {}
    for w in words:                      # bước 1: mọi chữ đều là đỉnh
        for ch in w:
            if ch not in adj:
                adj[ch] = []
                indeg[ch] = 0
    for w1, w2 in zip(words, words[1:]): # bước 2: cạnh từ các cặp liền kề
        kind, a, b = compare_pair(w1, w2)
        if kind == "invalid":
            return None
        if kind == "edge" and b not in adj[a]:
            adj[a].append(b)
            indeg[b] += 1
    return adj, indeg

for words in [ex1, ex3, ["abc", "ab"], ["ab", "adc"]]:
    g = build_graph(words)
    if g is None:
        print(f"{words}: tiền tố không hợp lệ -> None")
    else:
        adj, indeg = g
        print(f"{words}:\n    adj   = {adj}\n    indeg = {indeg}")

['wrt', 'wrf', 'er', 'ett', 'rftt']:
    adj   = {'w': ['e'], 'r': ['t'], 't': ['f'], 'f': [], 'e': ['r']}
    indeg = {'w': 0, 'r': 1, 't': 1, 'f': 1, 'e': 1}
['z', 'x', 'z']:
    adj   = {'z': ['x'], 'x': ['z']}
    indeg = {'z': 1, 'x': 1}
['abc', 'ab']: tiền tố không hợp lệ -> None
['ab', 'adc']:
    adj   = {'a': [], 'b': ['d'], 'd': [], 'c': []}
    indeg = {'a': 0, 'b': 0, 'd': 1, 'c': 0}


### Topological sort (sắp xếp tô-pô)

Một **topological order** của đồ thị có hướng là một cách xếp **tất cả** các đỉnh thành một hàng sao cho **mọi cạnh `u → v` đều chỉ từ trái sang phải** (`u` đứng trước `v`). Thuật toán tìm ra nó gọi là **topological sort**.

Ví dụ mặc đồ: `quần, tất, áo, giày, thắt lưng` là hợp lệ; `tất, áo, quần, thắt lưng, giày` cũng hợp lệ; nhưng `giày, quần, ...` thì **không** (cạnh `quần → giày` bị chỉ ngược từ phải sang trái).

Sau khi dựng đồ thị, bài Alien Dictionary **chính là**: *"tìm một topological order của đồ thị chữ cái; nếu không có (do chu trình) thì trả `""`"* — còn trường hợp tiền tố đã được xử lý ngay lúc dựng đồ thị.

Một đồ thị có thể có **nhiều** topological order (đó là lý do đề cho trả về bất kỳ), hoặc **không có** cái nào (khi có chu trình).

Hàm `is_topological_order(order, adj)` dưới đây kiểm tra: (1) `order` chứa mỗi đỉnh đúng một lần, (2) mọi cạnh chỉ từ trái sang phải. Nó dùng `pos[u]` = vị trí của `u` trong `order`.

In [12]:
def is_topological_order(order, adj):
    if len(order) != len(adj) or set(order) != set(adj):
        return False                      # thiếu, thừa hoặc lặp đỉnh
    pos = {u: i for i, u in enumerate(order)}
    return all(pos[u] < pos[v] for u in adj for v in adj[u])

print(is_topological_order(["quần", "tất", "áo", "giày", "thắt lưng"], dress))   # True
print(is_topological_order(["tất", "áo", "quần", "thắt lưng", "giày"], dress))   # True
print(is_topological_order(["giày", "quần", "tất", "áo", "thắt lưng"], dress))   # False
adj1, _ = build_graph(ex1)
print(is_topological_order(list("wertf"), adj1))                                 # True

True
True
False
True


## 7. Cách 2 — So sánh MỌI cặp từ + chọn dần chữ "tự do"

Từ mục 5, người mới thường nghĩ: *"Từ điển đã sắp xếp thì **mọi** cặp `i < j` đều phải có `words[i] <= words[j]`. Vậy cứ so sánh **tất cả** các cặp cho chắc, gom hết luật lại."*

Có tập luật rồi, xếp chữ thế nào? Ý tưởng tự nhiên — giống xếp thứ tự mặc đồ:

```text
remaining = mọi chữ cái (chưa xếp)
lặp cho tới khi remaining rỗng:
    tìm một chữ c trong remaining mà KHÔNG có luật x -> c với x vẫn còn trong remaining
    (tức là mọi chữ phải đứng trước c đều đã được xếp rồi)
    nếu không tìm được chữ nào như vậy -> các chữ còn lại "chặn nhau" thành vòng -> trả ""
    xếp c vào cuối đáp án, bỏ c khỏi remaining
```

Chữ `c` như vậy gọi là chữ **"tự do"** ở thời điểm đó: không còn ai bắt nó phải chờ. Đây đã là **topological sort** ở dạng thô sơ nhất!

Code dưới lưu luật thành các tuple `(a, b)`, rồi đưa vào `edge_set` (một `set` các tuple) để tra cứu `(x, c) in edge_set` nhanh. Ta chỉ **tra cứu** set, không duyệt nó, nên không lo thứ tự. Với `verbose=True`, mỗi vòng in ra các chữ còn lại, chữ nào đang bị chặn bởi ai (`r←e` nghĩa là `r` còn bị `e` chặn, vì có luật `e → r` mà `e` chưa xếp), và chữ được chọn.

In [13]:
def all_pairs_edges(words):
    # So sánh MỌI cặp i < j. Trả về list luật (không trùng) hoặc None nếu gặp tiền tố không hợp lệ
    edges = []
    for i in range(len(words)):
        for j in range(i + 1, len(words)):
            kind, a, b = compare_pair(words[i], words[j])
            if kind == "invalid":
                return None
            if kind == "edge" and (a, b) not in edges:
                edges.append((a, b))
    return edges

def alien_order_all_pairs(words, verbose=False):
    edges = all_pairs_edges(words)
    if edges is None:
        return ""
    edge_set = set(edges)
    remaining = unique_letters(words)
    order, rnd = [], 1
    while remaining:
        free, blocked = None, []
        for c in remaining:
            blockers = [x for x in remaining if (x, c) in edge_set]
            if blockers:
                blocked.append(f"{c}←{''.join(blockers)}")
            else:
                free = c                   # chữ tự do đầu tiên tìm được
                break
        if verbose:
            verdict = f"chọn {free!r}" if free is not None else "không có chữ tự do -> CHU TRÌNH"
            print(f"Vòng {rnd}: còn [{' '.join(remaining)}] | bị chặn: {', '.join(blocked) or '-'} | {verdict}")
        if free is None:
            return ""
        order.append(free)
        remaining.remove(free)
        rnd += 1
    return "".join(order)

print("Luật từ MỌI cặp của ex1:", all_pairs_edges(ex1))
print("Kết quả:", alien_order_all_pairs(ex1, verbose=True))
print()
print("Kết quả:", repr(alien_order_all_pairs(ex3, verbose=True)))

Luật từ MỌI cặp của ex1: [('t', 'f'), ('w', 'e'), ('w', 'r'), ('r', 't'), ('e', 'r')]
Vòng 1: còn [w r t f e] | bị chặn: - | chọn 'w'
Vòng 2: còn [r t f e] | bị chặn: r←e, t←r, f←t | chọn 'e'
Vòng 3: còn [r t f] | bị chặn: - | chọn 'r'
Vòng 4: còn [t f] | bị chặn: - | chọn 't'
Vòng 5: còn [f] | bị chặn: - | chọn 'f'
Kết quả: wertf

Vòng 1: còn [z x] | bị chặn: z←x, x←z | không có chữ tự do -> CHU TRÌNH
Kết quả: ''


### Vì sao chỉ cần so sánh các từ LIỀN KỀ?

Đọc output: so sánh mọi cặp của Example 1 cho **5** luật, trong khi so sánh liền kề (mục 5) chỉ cho **4**. Luật dư là `w → r` (từ cặp không liền kề `wrt` / `rftt`). Nhưng `w → r` **đã suy ra được** từ `w → e` và `e → r`! Luật từ cặp không liền kề luôn là "hệ quả" của các luật liền kề. Lý do là **tính bắc cầu** (transitivity) của thứ tự từ điển:

> Với một bảng chữ cái cố định, nếu `A <= B` và `B <= C` thì `A <= C`.

Ví dụ tiếng Anh: `"apple" <= "apply"` và `"apply" <= "banana"` ⇒ chắc chắn `"apple" <= "banana"`, không cần so lại.

Từ đó có chuỗi lập luận then chốt (σ là một bảng chữ cái bất kỳ):

1. `words` được sắp xếp theo σ ⟺ mọi cặp **liền kề** đúng thứ tự theo σ. Chiều ⇐ nhờ bắc cầu: `w0 <= w1 <= w2 <= ...` kéo theo `wi <= wj` với mọi `i < j`. (Trọng tài `is_sorted_by_order` cũng chỉ kiểm tra cặp liền kề — chính vì lý do này.)
2. Một cặp liền kề đúng thứ tự theo σ ⟺ cặp đó không "invalid" **và** σ đặt `a` trước `b` (với luật `a → b` của cặp, nếu có).
3. Vậy: **tập các bảng chữ cái hợp lệ = tập các topological order của đồ thị chỉ dựng từ cặp liền kề** (khi không có cặp invalid). Thêm cạnh từ cặp không liền kề **không làm thay đổi** tập này — chỉ tốn thêm công.

Một chi tiết tinh tế: với `["abc", "b", "ab"]`, cặp **không** liền kề `abc / ab` là invalid (tiền tố), còn các cặp liền kề cho `a → b` và `b → a` — một chu trình. Hai cách phát hiện lỗi khác nhau, nhưng **kết luận giống nhau**: không có đáp án.

Cell dưới **kiểm chứng bằng máy** trên 300 danh sách ngẫu nhiên: tập thứ tự hợp lệ tính bằng brute force, bằng "chỉ cạnh liền kề", và bằng "cạnh mọi cặp" phải **trùng khít**. Hàm sinh dữ liệu `random_words` dùng `words.sort(key=lambda w: [...])`: `lambda w: ...` là một hàm vô danh viết trên một dòng, còn `sort(key=f)` sắp xếp theo giá trị `f(w)` — ở đây là list thứ hạng theo một bảng chữ cái ẩn, đúng như mục 3. Nhờ vậy ta có đủ cả danh sách hợp lệ lẫn không hợp lệ (một danh sách chưa sắp xếp vẫn có thể tình cờ hợp lệ, nên tỉ lệ hợp lệ thực tế cao hơn một nửa — xem output).

In [14]:
adj1, _ = build_graph(ex1)
adjacent_edges = [(a, b) for a in adj1 for b in adj1[a]]
all_edges = all_pairs_edges(ex1)
print("Luật từ cặp liền kề:", adjacent_edges)
print("Luật từ mọi cặp    :", all_edges)
print("Luật dư            :", [e for e in all_edges if e not in adjacent_edges])

def valid_orders_by_edges(letters, edges):
    # Mọi hoán vị của letters thỏa tất cả các luật (a trước b)
    result = set()
    for perm in permutations(letters):
        pos = {c: i for i, c in enumerate(perm)}
        if all(pos[a] < pos[b] for a, b in edges):
            result.add("".join(perm))
    return result

def random_words(alphabet="abcde", max_n=5, max_len=3):
    alpha = random.sample(alphabet, random.randint(1, len(alphabet)))
    words = ["".join(random.choice(alpha) for _ in range(random.randint(1, max_len)))
             for _ in range(random.randint(1, max_n))]
    if random.random() < 0.5:                      # một nửa: sắp xếp theo một bảng chữ cái ẩn
        hidden = random.sample(alpha, len(alpha))
        words.sort(key=lambda w: [hidden.index(c) for c in w])
    return words

random.seed(7)
n_valid = 0
for _ in range(300):
    words = random_words()
    letters = unique_letters(words)
    brute = {"".join(p) for p in permutations(letters) if is_sorted_by_order(words, p)}
    g = build_graph(words)
    adj_only = set() if g is None else valid_orders_by_edges(letters, [(a, b) for a in g[0] for b in g[0][a]])
    e_all = all_pairs_edges(words)
    all_pairs = set() if e_all is None else valid_orders_by_edges(letters, e_all)
    assert brute == adj_only == all_pairs, words
    n_valid += bool(brute)
print(f"300 danh sách ngẫu nhiên ({n_valid} hợp lệ): brute == chỉ cạnh liền kề == cạnh mọi cặp -> PASS")

Luật từ cặp liền kề: [('w', 'e'), ('r', 't'), ('t', 'f'), ('e', 'r')]
Luật từ mọi cặp    : [('t', 'f'), ('w', 'e'), ('w', 'r'), ('r', 't'), ('e', 'r')]
Luật dư            : [('w', 'r')]
300 danh sách ngẫu nhiên (239 hợp lệ): brute == chỉ cạnh liền kề == cạnh mọi cặp -> PASS


### Nút thắt của cách 2

Cách 2 đúng, nhưng tốn công ở hai chỗ:

1. **So sánh mọi cặp**: có `n(n-1)/2` cặp, mỗi cặp tốn tới `L` phép so sánh ký tự → `O(n² · L)`. Với `n = L = 100` là cỡ `4950 × 100 ≈ 5 · 10^5` phép — vẫn chạy được với constraints nhỏ của bài này, nhưng **lãng phí khoảng 50 lần** so với chỉ so các cặp liền kề (`99 × 100 ≈ 10^4`). Cell dưới đếm thật trên một input xấu (100 từ dài 100, chung 98 chữ đầu nên phải so gần hết mới thấy chỗ khác).
2. **Chọn chữ tự do bằng cách quét lại từ đầu**: mỗi vòng quét mọi chữ còn lại, với mỗi chữ lại quét mọi "người chặn" có thể → `O(k²)` mỗi vòng, `k` vòng → `O(k³)`. Ta đang **tính lại từ đầu** một thông tin hoàn toàn có thể **cập nhật dần**: *"chữ `c` còn bao nhiêu người chặn?"* — đó chính là **in-degree**! Khi xếp xong một chữ `u`, chỉ những chữ mà `u` chỉ tới mới bớt đi một người chặn.

Hai cải tiến này dẫn thẳng tới **cách 3 (Kahn)**: chỉ dùng cặp liền kề + giữ `indeg` cập nhật dần + một **hàng đợi** chứa các chữ đang tự do.

In [15]:
def count_char_comparisons(pairs):
    total = 0
    for w1, w2 in pairs:
        for a, b in zip(w1, w2):
            total += 1
            if a != b:
                break
    return total

letters26 = "abcdefghijklmnopqrstuvwxyz"
worst = ["a" * 98 + x + y for x in letters26 for y in letters26][:100]   # 100 từ dài 100
n = len(worst)
adjacent_pairs = list(zip(worst, worst[1:]))
all_pairs_list = [(worst[i], worst[j]) for i in range(n) for j in range(i + 1, n)]
print("n =", n, "| L =", len(worst[0]))
print(f"Cặp liền kề: {len(adjacent_pairs):5} cặp, {count_char_comparisons(adjacent_pairs):8,} phép so sánh ký tự")
print(f"Mọi cặp    : {len(all_pairs_list):5} cặp, {count_char_comparisons(all_pairs_list):8,} phép so sánh ký tự")
ans = alien_order_all_pairs(worst)
print("Cách 2 vẫn ra đáp án đúng:", ans, is_sorted_by_order(worst, ans))

n = 100 | L = 100
Cặp liền kề:    99 cặp,    9,897 phép so sánh ký tự
Mọi cặp    :  4950 cặp,  491,256 phép so sánh ký tự
Cách 2 vẫn ra đáp án đúng: abcdefghijklmnopqrstuvwxyz True


## 8. Nền tảng 4 — Hàng đợi (queue) và `collections.deque`

**Hàng đợi** (queue) hoạt động như xếp hàng mua vé: ai đến trước được phục vụ trước — **FIFO** (First In, First Out). Hai thao tác chính:

- **enqueue**: thêm vào **cuối** hàng;
- **dequeue**: lấy ra từ **đầu** hàng.

Trong Python ta dùng `deque` (double-ended queue — hàng đợi hai đầu) từ module `collections`:

```python
q = deque(["a", "b"])
q.append("c")      # thêm vào cuối  -> deque(['a', 'b', 'c'])
x = q.popleft()    # lấy từ đầu     -> x = 'a', còn deque(['b', 'c'])
```

Vì sao không dùng `list` với `lst.pop(0)`? Vì `pop(0)` phải **dời mọi phần tử còn lại** sang trái một ô → tốn `O(độ dài)`; còn `deque.popleft()` chỉ `O(1)`. Với bài này (≤ 26 chữ) khác biệt nhỏ, nhưng thói quen đúng sẽ cứu bạn ở các bài đồ thị lớn (10^5 đỉnh).

Kiểm tra hàng rỗng: `while q:` — một deque rỗng được coi là `False`.

In [16]:
q = deque()
for ch in "xyz":
    q.append(ch)
    print("append", ch, "->", list(q))
while q:
    ch = q.popleft()
    print("popleft ->", ch, "| còn lại:", list(q))
print("Hàng rỗng -> bool(q) =", bool(q))

append x -> ['x']
append y -> ['x', 'y']
append z -> ['x', 'y', 'z']
popleft -> x | còn lại: ['y', 'z']
popleft -> y | còn lại: ['z']
popleft -> z | còn lại: []
Hàng rỗng -> bool(q) = False


## 9. Cách 3 (chính) — Thuật toán Kahn: BFS theo in-degree

Tinh chỉnh từ cách 2: thay vì quét lại tìm chữ tự do, ta **duy trì** hai thứ:

- `indeg[v]` = số chữ **chưa được xếp** mà còn bắt `v` phải chờ;
- `queue` = các chữ **đang tự do** (in-degree 0) nhưng chưa xếp.

```text
1. Tính indeg cho mọi đỉnh; cho mọi đỉnh có indeg 0 vào queue.
2. Lặp khi queue còn phần tử:
     u = queue.popleft();  xếp u vào cuối order
     với mỗi v mà u -> v:
         indeg[v] -= 1          # u đã xếp xong, v bớt một người phải chờ
         nếu indeg[v] == 0:     # v vừa trở nên tự do
             queue.append(v)
3. Nếu len(order) == số đỉnh: order là một topological order.
   Ngược lại: các đỉnh còn lại không bao giờ về 0 -> có chu trình.
```

Tại sao gọi là **BFS** (Breadth-First Search — tìm kiếm theo chiều rộng)? Vì nó dùng hàng đợi và "lan" dần từ các đỉnh nguồn ra ngoài theo từng lớp, giống BFS thông thường; chỉ khác là một đỉnh chỉ được vào hàng khi **mọi** cạnh đi vào nó đã được "gỡ".

Vì sao đúng (trực giác): một đỉnh chỉ vào `queue` khi in-degree về 0, tức là **mọi** đỉnh phải đứng trước nó **đã nằm trong `order`**. Vậy mỗi cạnh `u → v` đều có `u` được xếp trước `v`.

Thử trước trên đồ thị mặc đồ (chưa đụng tới chữ cái), rồi trên một đồ thị **có chu trình**. Hàm `kahn_order(adj, indeg, verbose)` trả về list `order` (có thể **thiếu** đỉnh nếu có chu trình); nó sao chép `indeg` bằng `dict(indeg)` để không làm hỏng dữ liệu của người gọi.

In [17]:
def kahn_order(adj, indeg, verbose=False):
    indeg = dict(indeg)                            # bản sao: không sửa dict của người gọi
    queue = deque(u for u in adj if indeg[u] == 0)
    order = []
    if verbose:
        print("  Ban đầu queue =", list(queue))
    while queue:
        u = queue.popleft()
        order.append(u)
        freed = []
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
                freed.append(v)
        if verbose:
            print(f"  lấy {u!r:12} -> tự do mới: {freed}; queue = {list(queue)}")
    return order

print("Đồ thị mặc đồ:")
print("order =", kahn_order(dress, compute_indegree(dress), verbose=True))

cyc = {"d": ["a"], "a": ["b"], "b": ["c"], "c": ["a"]}      # a -> b -> c -> a là chu trình
print("\nĐồ thị có chu trình (d -> a, a -> b -> c -> a):")
print("order =", kahn_order(cyc, compute_indegree(cyc), verbose=True), "| số đỉnh =", len(cyc))

Đồ thị mặc đồ:
  Ban đầu queue = ['quần', 'tất', 'áo']
  lấy 'quần'       -> tự do mới: []; queue = ['tất', 'áo']
  lấy 'tất'        -> tự do mới: ['giày']; queue = ['áo', 'giày']
  lấy 'áo'         -> tự do mới: ['thắt lưng']; queue = ['giày', 'thắt lưng']
  lấy 'giày'       -> tự do mới: []; queue = ['thắt lưng']
  lấy 'thắt lưng'  -> tự do mới: []; queue = []
order = ['quần', 'tất', 'áo', 'giày', 'thắt lưng']

Đồ thị có chu trình (d -> a, a -> b -> c -> a):
  Ban đầu queue = ['d']
  lấy 'd'          -> tự do mới: []; queue = []
order = ['d'] | số đỉnh = 4


### Đọc output và ghép thành `alien_order_kahn`

- **Đồ thị mặc đồ**: lấy `quần` xong, `giày` và `thắt lưng` chỉ giảm từ 2 xuống 1 — **chưa** tự do (vẫn chờ `tất` và `áo`), nên "tự do mới" là `[]`. Lấy `tất` → `giày` về 0 → vào hàng. Lấy `áo` → `thắt lưng` về 0 → vào hàng. Kết quả `quần, tất, áo, giày, thắt lưng` — một topological order hợp lệ (đã kiểm ở mục 6).
- **Đồ thị có chu trình**: chỉ `d` có in-degree 0. Lấy `d` làm `a` giảm từ 2 xuống 1, nhưng `a` vẫn chờ `c`, `c` chờ `b`, `b` chờ `a` — **chờ nhau vòng tròn**, không ai về 0. Hàng đợi cạn khi `order` mới có 1/4 đỉnh → **phát hiện chu trình chỉ bằng một phép so sánh độ dài**.

Lời giải Alien Dictionary bằng Kahn chỉ là ghép hai mảnh đã có:

```text
graph = build_graph(words)          # mục 6: cạnh từ cặp liền kề + kiểm tra tiền tố
nếu graph là None -> ""             # tiền tố đứng sau
order = kahn_order(adj, indeg)      # mục 9
nếu len(order) < số chữ cái -> ""   # có chu trình
ngược lại -> "".join(order)
```

In [18]:
def alien_order_kahn(words):
    graph = build_graph(words)
    if graph is None:
        return ""                       # tiền tố không hợp lệ
    adj, indeg = graph
    order = kahn_order(adj, indeg)
    if len(order) < len(adj):
        return ""                       # có chu trình
    return "".join(order)

for words in [ex1, ex2, ex3, ["abc", "ab"], ["ab", "adc"], ["z", "z"], ["ab", "ba"]]:
    print(f"{str(words):36} -> {alien_order_kahn(words)!r}")

['wrt', 'wrf', 'er', 'ett', 'rftt']  -> 'wertf'
['z', 'x']                           -> 'zx'
['z', 'x', 'z']                      -> ''
['abc', 'ab']                        -> ''
['ab', 'adc']                        -> 'abcd'
['z', 'z']                           -> 'z'
['ab', 'ba']                         -> 'ab'


## 10. Trace Kahn trên ví dụ có rẽ nhánh

Example 1 là một "chuỗi thẳng" nên hàng đợi lúc nào cũng chỉ có 1 chữ — chưa thú vị. Ta dùng một ví dụ tự tạo có rẽ nhánh và một chữ có in-degree 2:

```python
words = ["xa", "xb", "ya", "yc", "c", "z"]
```

Cặp liền kề → luật: `xa/xb`: `a → b`; `xb/ya`: `x → y`; `ya/yc`: `a → c`; `yc/c`: `y → c`; `c/z`: `c → z`.

```text
x ──► y ──► c ──► z
            ▲
a ──────────┘
│
▼
b
```

Chữ theo thứ tự xuất hiện: `x a b y c z`. In-degree ban đầu: `x:0  a:0  b:1  y:1  c:2  z:1`.

| Bước | Lấy ra `u` | Cạnh `u → v`: indeg mới | Vào queue | queue sau bước | order |
|---|---|---|---|---|---|
| 0 | — | — | `x`, `a` | `[x, a]` | `""` |
| 1 | `x` | `y: 1→0` | `y` | `[a, y]` | `x` |
| 2 | `a` | `b: 1→0`, `c: 2→1` | `b` | `[y, b]` | `xa` |
| 3 | `y` | `c: 1→0` | `c` | `[b, c]` | `xay` |
| 4 | `b` | (không có cạnh ra) | — | `[c]` | `xayb` |
| 5 | `c` | `z: 1→0` | `z` | `[z]` | `xaybc` |
| 6 | `z` | (không có cạnh ra) | — | `[]` | `xaybcz` |

Điểm đáng chú ý ở bước 2: lấy `a` chỉ làm `c` giảm từ 2 xuống 1 — `c` còn phải chờ `y`. Tới bước 3, khi `y` đã được xếp, `c` mới tự do. Đó chính là ý nghĩa của in-degree: **đếm số người mình còn phải chờ**.

Cell trace còn chạy thêm hai trường hợp trả `""` — hãy **dự đoán trước** khi đọc output:

- **(a) Chu trình kèm chữ tự do** — `["a", "b", "c", "b"]`: các cặp cho `a → b`, `b → c`, `c → b`; `b` và `c` chặn nhau. Queue ban đầu chỉ có `a`; lấy `a` làm `b` giảm 2 → 1 (vẫn chờ `c`); queue cạn khi mới xếp 1/3 chữ → `""`. Nếu quên so độ dài ở cuối, ta sẽ trả `"a"` — **sai** (lỗi thường gặp ở mục 16).
- **(b) Tiền tố đứng sau** — `["abc", "ab"]`: `build_graph` trả `None` ngay, Kahn không cần chạy.

In [19]:
def trace_kahn(words):
    print("words =", words)
    graph = build_graph(words)
    if graph is None:
        print("Gặp tiền tố không hợp lệ khi dựng đồ thị -> ''")
        return ""
    adj, indeg = graph
    print("adj   =", adj)
    print("indeg =", indeg)
    queue = deque(c for c in adj if indeg[c] == 0)
    order = []
    print(f"Bước 0: queue = {list(queue)}")
    step = 1
    while queue:
        u = queue.popleft()
        order.append(u)
        changes, pushed = [], []
        for v in adj[u]:
            indeg[v] -= 1
            changes.append(f"{v}:{indeg[v] + 1}->{indeg[v]}")
            if indeg[v] == 0:
                queue.append(v)
                pushed.append(v)
        print(f"Bước {step}: lấy {u} | {', '.join(changes) or '(không có cạnh ra)':18} | "
              f"vào {pushed} | queue = {list(queue)} | order = {''.join(order)!r}")
        step += 1
    if len(order) < len(adj):
        stuck = {c: d for c, d in indeg.items() if d > 0}
        print(f"Queue cạn nhưng mới xếp {len(order)}/{len(adj)} chữ; còn kẹt {stuck} -> CHU TRÌNH -> ''")
        return ""
    print("Kết quả:", "".join(order))
    return "".join(order)

trace_kahn(["xa", "xb", "ya", "yc", "c", "z"])
print()
trace_kahn(["a", "b", "c", "b"])
print()
_ = trace_kahn(["abc", "ab"])   # gán vào _ để Jupyter không in thêm giá trị trả về

words = ['xa', 'xb', 'ya', 'yc', 'c', 'z']
adj   = {'x': ['y'], 'a': ['b', 'c'], 'b': [], 'y': ['c'], 'c': ['z'], 'z': []}
indeg = {'x': 0, 'a': 0, 'b': 1, 'y': 1, 'c': 2, 'z': 1}
Bước 0: queue = ['x', 'a']
Bước 1: lấy x | y:1->0             | vào ['y'] | queue = ['a', 'y'] | order = 'x'
Bước 2: lấy a | b:1->0, c:2->1     | vào ['b'] | queue = ['y', 'b'] | order = 'xa'
Bước 3: lấy y | c:1->0             | vào ['c'] | queue = ['b', 'c'] | order = 'xay'
Bước 4: lấy b | (không có cạnh ra) | vào [] | queue = ['c'] | order = 'xayb'
Bước 5: lấy c | z:1->0             | vào ['z'] | queue = ['z'] | order = 'xaybc'
Bước 6: lấy z | (không có cạnh ra) | vào [] | queue = [] | order = 'xaybcz'
Kết quả: xaybcz

words = ['a', 'b', 'c', 'b']
adj   = {'a': ['b'], 'b': ['c'], 'c': ['b']}
indeg = {'a': 0, 'b': 2, 'c': 1}
Bước 0: queue = ['a']
Bước 1: lấy a | b:2->1             | vào [] | queue = [] | order = 'a'
Queue cạn nhưng mới xếp 1/3 chữ; còn kẹt {'b': 1, 'c': 1} -> CHU TRÌNH -> ''

words = ['abc',

## 11. Nền tảng 5 — Đệ quy (recursion) và DFS trên đồ thị

**Đệ quy** là khi một hàm **gọi lại chính nó** với bài toán nhỏ hơn, và có **điểm dừng** (base case) để không gọi mãi. Mỗi lời gọi được đặt lên **call stack** (ngăn xếp lời gọi): lời gọi vào **sau** sẽ xong **trước**, rồi mới quay về lời gọi bên ngoài.

**DFS** (Depth-First Search — tìm kiếm theo chiều sâu) trên đồ thị: từ một đỉnh, đi theo một cạnh **sâu hết mức** có thể; hết đường mới **quay lui** (backtrack) và thử cạnh khác. Viết bằng đệ quy rất tự nhiên:

```python
def dfs(u):
    visited.add(u)
    for v in adj[u]:
        if v not in visited:
            dfs(v)
    post.append(u)      # u "xong" khi mọi đỉnh đi được từ u đã xong
```

Dòng `post.append(u)` đặt **sau** vòng lặp, nên `u` chỉ được ghi khi đã xử lý xong **toàn bộ** các đỉnh mà nó dẫn tới. Thứ tự ghi này gọi là **post-order** (thứ tự "xong sau").

Trong code, `dfs` được viết **lồng** bên trong một hàm khác (nested function) nên dùng được trực tiếp các biến `visited`, `post`, `adj` của hàm ngoài, không cần truyền qua tham số.

Cell dưới gồm: (1) một ví dụ đệ quy siêu nhỏ `countdown` để thấy thứ tự "vào — ra" trên call stack; (2) DFS trên đồ thị mặc đồ, in thụt lề theo độ sâu và ghi lại post-order.

In [20]:
def countdown(k, depth=0):
    pad = "    " * depth
    print(f"{pad}vào countdown({k})")
    if k > 0:                         # base case: k == 0 thì không gọi tiếp
        countdown(k - 1, depth + 1)
    print(f"{pad}ra  countdown({k})")

countdown(2)

def dfs_postorder(adj):
    visited, post = set(), []
    def dfs(u, depth):
        visited.add(u)
        print("    " * depth + f"vào {u}")
        for v in adj[u]:
            if v not in visited:
                dfs(v, depth + 1)
        post.append(u)
        print("    " * depth + f"xong {u} -> post = {post}")
    for u in adj:
        if u not in visited:
            dfs(u, 0)
    return post

print()
post = dfs_postorder(dress)
print("post-order      :", post)
print("đảo ngược post  :", post[::-1])
print("là topological order?", is_topological_order(post[::-1], dress))

vào countdown(2)
    vào countdown(1)
        vào countdown(0)
        ra  countdown(0)
    ra  countdown(1)
ra  countdown(2)

vào quần
    vào giày
    xong giày -> post = ['giày']
    vào thắt lưng
    xong thắt lưng -> post = ['giày', 'thắt lưng']
xong quần -> post = ['giày', 'thắt lưng', 'quần']
vào tất
xong tất -> post = ['giày', 'thắt lưng', 'quần', 'tất']
vào áo
xong áo -> post = ['giày', 'thắt lưng', 'quần', 'tất', 'áo']
post-order      : ['giày', 'thắt lưng', 'quần', 'tất', 'áo']
đảo ngược post  : ['áo', 'tất', 'quần', 'thắt lưng', 'giày']
là topological order? True


### Vì sao đảo ngược post-order lại là topological order? Và vì sao cần 3 màu?

Xét một cạnh `u → v` bất kỳ. Khi DFS đang ở `u` và nhìn sang `v`, có 3 khả năng:

1. `v` **chưa thăm** → ta gọi `dfs(v)`; `v` xong **trước** khi `u` xong → `v` nằm trước `u` trong `post`.
2. `v` **đã xong** từ trước → `v` đã có trong `post`, còn `u` sẽ được thêm sau → vẫn `v` trước `u`.
3. `v` **đang thăm dở** — tức `v` nằm trên đường đi hiện tại từ gốc DFS tới `u` (đang trên call stack). Khi đó có đường `v → ... → u`, cộng thêm cạnh `u → v` → **chu trình**!

Nếu không có chu trình thì chỉ xảy ra (1) và (2): với mọi cạnh `u → v`, `v` xong trước `u`. Đảo ngược `post` → `u` đứng trước `v` với **mọi** cạnh. Đó chính là topological order (output trên: `áo, tất, quần, thắt lưng, giày` — hợp lệ).

Vấn đề: chỉ dùng `visited` (2 trạng thái) thì **không phân biệt được** (2) "đã xong — an toàn" với (3) "đang thăm dở — chu trình". Ở ví dụ mặc đồ, khi `tất` nhìn sang `giày` thì `giày` đã xong (trường hợp 2) — vô hại. Vì vậy ta cần **3 màu**:

| Màu | Ý nghĩa | Đang ở `u`, gặp `v` có màu này thì làm gì? |
|---|---|---|
| **WHITE** (trắng) | chưa thăm | gọi `dfs(v)` |
| **GRAY** (xám) | đang thăm dở — đang nằm trên call stack | **chu trình** → cả bài trả `""` |
| **BLACK** (đen) | đã xong hoàn toàn | bỏ qua, an toàn |

Một đỉnh đổi màu: WHITE → GRAY khi **vào** `dfs`, GRAY → BLACK khi **ra** (đúng lúc `post.append`).

## 12. Cách 4 — DFS 3 màu (white / gray / black)

- `color[c]` khởi tạo WHITE cho mọi chữ (đồ thị dựng bằng `build_graph` như cũ; DFS không cần `indeg`).
- `dfs(u)` trả về `True` nếu không phát hiện chu trình trong phần đồ thị đi được từ `u`, `False` nếu có. Gặp `False` ở đâu thì "truyền" `False` ngược lên ngay.
- Gọi `dfs` lần lượt từ mọi chữ còn WHITE — đồ thị có thể gồm nhiều "mảnh" rời nhau, như `["ab", "adc"]` (chữ `a`, `c` đứng riêng).
- Kết quả là `post` đảo ngược.

Độ sâu đệ quy tối đa = số chữ cái ≤ 26, nên không lo vượt giới hạn đệ quy mặc định của Python (1000).

In [21]:
WHITE, GRAY, BLACK = 0, 1, 2

def alien_order_dfs(words):
    graph = build_graph(words)
    if graph is None:
        return ""
    adj, _ = graph                      # DFS không cần in-degree
    color = {c: WHITE for c in adj}
    post = []

    def dfs(u):
        color[u] = GRAY
        for v in adj[u]:
            if color[v] == GRAY:        # quay lại đỉnh đang thăm dở -> chu trình
                return False
            if color[v] == WHITE and not dfs(v):
                return False
        color[u] = BLACK                # BLACK thì bỏ qua (an toàn)
        post.append(u)
        return True

    for c in adj:
        if color[c] == WHITE and not dfs(c):
            return ""
    return "".join(reversed(post))

for words in [ex1, ex2, ex3, ["abc", "ab"], ["ab", "adc"], ["z", "z"], ["xa", "xb", "ya", "yc", "c", "z"]]:
    print(f"{str(words):36} DFS -> {alien_order_dfs(words)!r:9} | Kahn -> {alien_order_kahn(words)!r}")

['wrt', 'wrf', 'er', 'ett', 'rftt']  DFS -> 'wertf'   | Kahn -> 'wertf'
['z', 'x']                           DFS -> 'zx'      | Kahn -> 'zx'
['z', 'x', 'z']                      DFS -> ''        | Kahn -> ''
['abc', 'ab']                        DFS -> ''        | Kahn -> ''
['ab', 'adc']                        DFS -> 'cbda'    | Kahn -> 'abcd'
['z', 'z']                           DFS -> 'z'       | Kahn -> 'z'
['xa', 'xb', 'ya', 'yc', 'c', 'z']   DFS -> 'abxycz'  | Kahn -> 'xaybcz'


### Trace DFS 3 màu

Cùng ví dụ rẽ nhánh `["xa", "xb", "ya", "yc", "c", "z"]`, danh sách kề `x:[y]  a:[b, c]  b:[]  y:[c]  c:[z]  z:[]`, duyệt gốc theo thứ tự `x a b y c z`:

| Sự kiện | Màu thay đổi | `post` |
|---|---|---|
| vào `x` (gốc) | x: GRAY | `""` |
| `x → y`: y WHITE → đi sâu, vào `y` | y: GRAY | |
| `y → c`: c WHITE → đi sâu, vào `c` | c: GRAY | |
| `c → z`: z WHITE → đi sâu, vào `z` | z: GRAY | |
| xong `z` | z: BLACK | `z` |
| xong `c`, rồi `y`, rồi `x` | c, y, x: BLACK | `zcyx` |
| vào `a` (gốc mới vì `a` còn WHITE) | a: GRAY | |
| `a → b`: WHITE → vào `b`, xong `b` | b: BLACK | `zcyxb` |
| `a → c`: c đã BLACK → **bỏ qua** (an toàn, không phải chu trình) | | |
| xong `a` | a: BLACK | `zcyxba` |

Đảo ngược `post`: `"abxycz"`. Khác với Kahn (`"xaybcz"`) nhưng **cũng hợp lệ**: `a` trước `b`, `a` trước `c`, `x` trước `y`, `y` trước `c`, `c` trước `z` đều thỏa. Hai thuật toán đúng có thể cho hai đáp án khác nhau — lại thêm lý do phải test bằng hàm kiểm tra hợp lệ.

Trace thứ hai là Example 3 `["z","x","z"]`: vào `z` (GRAY) → `z → x`: x WHITE, vào `x` (GRAY) → `x → z`: z đang **GRAY** → **chu trình** → `""`.

In [22]:
def trace_dfs(words):
    print("words =", words)
    graph = build_graph(words)
    if graph is None:
        print("Tiền tố không hợp lệ -> ''")
        return ""
    adj, _ = graph
    name = {WHITE: "WHITE", GRAY: "GRAY", BLACK: "BLACK"}
    color = {c: WHITE for c in adj}
    post = []

    def dfs(u, depth):
        pad = "    " * depth
        color[u] = GRAY
        print(f"{pad}vào {u} (GRAY)")
        for v in adj[u]:
            status = f"{pad}  {u} -> {v}: {v} đang {name[color[v]]}"
            if color[v] == GRAY:
                print(status, "=> CHU TRÌNH!")
                return False
            if color[v] == BLACK:
                print(status, "=> bỏ qua")
                continue
            print(status, "=> đi sâu")
            if not dfs(v, depth + 1):
                return False
        color[u] = BLACK
        post.append(u)
        print(f"{pad}xong {u} (BLACK), post = {''.join(post)!r}")
        return True

    for c in adj:
        if color[c] == WHITE and not dfs(c, 0):
            print("Kết quả: ''")
            return ""
    ans = "".join(reversed(post))
    print("Đảo ngược post ->", repr(ans))
    return ans

trace_dfs(["xa", "xb", "ya", "yc", "c", "z"])
print()
_ = trace_dfs(ex3)

words = ['xa', 'xb', 'ya', 'yc', 'c', 'z']
vào x (GRAY)
  x -> y: y đang WHITE => đi sâu
    vào y (GRAY)
      y -> c: c đang WHITE => đi sâu
        vào c (GRAY)
          c -> z: z đang WHITE => đi sâu
            vào z (GRAY)
            xong z (BLACK), post = 'z'
        xong c (BLACK), post = 'zc'
    xong y (BLACK), post = 'zcy'
xong x (BLACK), post = 'zcyx'
vào a (GRAY)
  a -> b: b đang WHITE => đi sâu
    vào b (GRAY)
    xong b (BLACK), post = 'zcyxb'
  a -> c: c đang BLACK => bỏ qua
xong a (BLACK), post = 'zcyxba'
Đảo ngược post -> 'abxycz'

words = ['z', 'x', 'z']
vào z (GRAY)
  z -> x: x đang WHITE => đi sâu
    vào x (GRAY)
      x -> z: z đang GRAY => CHU TRÌNH!
Kết quả: ''


## 13. Code nộp LeetCode — cách chính (Kahn)

Bản nộp gói mọi thứ trong một method, không in debug. Vài chỗ viết gọn cần giải thích:

- `adj = {c: [] for w in words for c in w}` — dict comprehension có **hai vòng `for` lồng nhau**, đọc từ trái sang phải giống hệt:

  ```python
  for w in words:
      for c in w:
          adj[c] = []
  ```

  Chữ trùng chỉ bị ghi đè bằng một list rỗng mới — vô hại vì lúc này chưa có cạnh nào. Mọi chữ xuất hiện đều có mặt → không bao giờ "rơi" chữ.
- `zip(words, words[1:])` — các cặp liền kề (mục 2).
- `for a, b in zip(w1, w2): ... break` + `else:` — mẫu `for ... else`: khối `else` chỉ chạy khi **không** tìm thấy chữ khác nhau; lúc đó nếu `len(w1) > len(w2)` là tiền tố không hợp lệ.
- `if b not in adj[a]` — chống cạnh trùng để `indeg` không bị đếm dư.
- Dòng cuối: số chữ đã xếp phải bằng số chữ cái, nếu không là có chu trình.

Trên LeetCode, `List` và `deque` đã được import sẵn; trong notebook ta đã import ở cell đầu.

In [23]:
class Solution:
    def alienOrder(self, words: List[str]) -> str:
        # 1) Mọi chữ xuất hiện đều là một đỉnh
        adj = {c: [] for w in words for c in w}
        indeg = {c: 0 for c in adj}

        # 2) Rút cạnh từ các cặp từ liền kề
        for w1, w2 in zip(words, words[1:]):
            for a, b in zip(w1, w2):
                if a != b:
                    if b not in adj[a]:          # chống cạnh trùng
                        adj[a].append(b)
                        indeg[b] += 1
                    break                        # chỉ chữ khác nhau ĐẦU TIÊN có ý nghĩa
            else:
                if len(w1) > len(w2):            # w2 là tiền tố thật sự của w1
                    return ""

        # 3) Kahn: lấy dần các chữ có in-degree 0
        queue = deque(c for c in adj if indeg[c] == 0)
        order = []
        while queue:
            c = queue.popleft()
            order.append(c)
            for nxt in adj[c]:
                indeg[nxt] -= 1
                if indeg[nxt] == 0:
                    queue.append(nxt)

        # 4) Thiếu chữ = có chu trình
        return "".join(order) if len(order) == len(adj) else ""


sol = Solution()
print(repr(sol.alienOrder(["wrt", "wrf", "er", "ett", "rftt"])))   # 'wertf'
print(repr(sol.alienOrder(["z", "x"])))                            # 'zx'
print(repr(sol.alienOrder(["z", "x", "z"])))                       # ''

'wertf'
'zx'
''


### Test bằng hàm kiểm tra tính hợp lệ (không so chuỗi cố định)

Với `["ab", "adc"]` có 12 đáp án đúng. Nếu test `assert ans == "abcd"`, một lời giải đúng khác (ví dụ DFS trả `"cbda"`) sẽ bị đánh trượt oan. Thay vào đó ta kiểm tra **tính chất** của đáp án:

- Nếu đáp án là `""`: phải **thật sự không tồn tại** thứ tự hợp lệ. Với input ít chữ (≤ 8) ta hỏi trọng tài brute force; với input lớn ta truyền kỳ vọng `expect_valid` mà ta biết chắc vì tự xây input.
- Nếu đáp án khác `""`: (1) là hoán vị của **đúng** tập chữ cái xuất hiện — không thiếu, không thừa, không lặp; (2) `is_sorted_by_order(words, ans)` phải `True`.

Bộ test `TESTS` gồm ví dụ đề và các ca dễ sai: tiền tố đứng sau / đứng trước, một từ, từ trùng nhau, chỉ so chữ khác đầu tiên (`["ab","ba"]`), đồ thị nhiều mảnh rời, chu trình kèm chữ tự do, cạnh trùng, chu trình 3 chữ, và hai input lớn 26 chữ cái. Input lớn "không hợp lệ" được tạo bằng cách chép từ đầu tiên xuống cuối: nếu danh sách sắp xếp được thì phải có `w0 <= ... <= w98 <= w0`, buộc mọi từ bằng nhau — vô lý vì các từ khác nhau.

In [24]:
def is_valid_alien_answer(words, ans, expect_valid=None):
    letters = set("".join(words))
    if expect_valid is None:                      # tự hỏi trọng tài brute force (chỉ khi ít chữ)
        assert len(letters) <= 8, "quá nhiều chữ cho brute force, hãy truyền expect_valid"
        expect_valid = alien_order_brute(words) != ""
    if ans == "":
        return not expect_valid
    return (expect_valid
            and len(ans) == len(letters) and set(ans) == letters
            and is_sorted_by_order(words, ans))

random.seed(269)
hidden26 = random.sample(letters26, 26)
big = ["".join(random.choice(letters26) for _ in range(random.randint(1, 100))) for _ in range(100)]
big.sort(key=lambda w: [hidden26.index(c) for c in w])   # sắp theo bảng chữ cái ẩn -> chắc chắn hợp lệ
big_bad = big[:99] + [big[0]]                             # chép từ đầu xuống cuối -> chắc chắn không hợp lệ

TESTS = [
    ("Example 1", ex1, None),
    ("Example 2", ex2, None),
    ("Example 3 (chu trình)", ex3, None),
    ("tiền tố đứng sau", ["abc", "ab"], None),
    ("tiền tố đứng trước", ["ab", "abc"], None),
    ("một từ", ["abc"], None),
    ("từ trùng nhau", ["z", "z"], None),
    ("chỉ so chữ khác đầu tiên", ["ab", "ba"], None),
    ("nhiều mảnh rời", ["ab", "adc"], None),
    ("chu trình + chữ tự do", ["a", "b", "c", "b"], None),
    ("cạnh trùng", ["ab", "ac", "bb", "bc"], None),
    ("chu trình 3 chữ", ["a", "b", "c", "a"], None),
    ("100 từ, bảng ẩn 26 chữ", big, True),
    ("như trên, chép từ đầu xuống cuối", big_bad, False),
]

def run_tests(solver):
    for name, words, expect_valid in TESTS:
        ans = solver(words)
        assert is_valid_alien_answer(words, ans, expect_valid), (name, words, ans)
        shown = ans if len(ans) <= 12 else ans[:12] + "..."
        print(f"PASS {name:34} -> {shown!r}")

run_tests(Solution().alienOrder)

PASS Example 1                          -> 'wertf'
PASS Example 2                          -> 'zx'
PASS Example 3 (chu trình)              -> ''
PASS tiền tố đứng sau                   -> ''
PASS tiền tố đứng trước                 -> 'abc'
PASS một từ                             -> 'abc'
PASS từ trùng nhau                      -> 'z'
PASS chỉ so chữ khác đầu tiên           -> 'ab'
PASS nhiều mảnh rời                     -> 'abcd'
PASS chu trình + chữ tự do              -> ''
PASS cạnh trùng                         -> 'abc'
PASS chu trình 3 chữ                    -> ''
PASS 100 từ, bảng ẩn 26 chữ             -> 'bactjgkvqxfm...'
PASS như trên, chép từ đầu xuống cuối   -> ''


### Bản LeetCode thứ hai — DFS 3 màu, và đối chiếu chéo ngẫu nhiên

Nếu thích đệ quy, bản DFS dưới đây cũng nộp được (cùng phần rút cạnh, thay phần Kahn bằng DFS 3 màu). Trong điều kiện `color[v] == GRAY or (color[v] == WHITE and not dfs(v))`, Python tính **ngắn mạch** (short-circuit): nếu vế trái đúng thì không tính vế phải; nếu `v` là BLACK thì cả hai vế sai → bỏ qua `v`.

Sau đó ta **đối chiếu chéo** mọi cách đã học (cách 2, cách 3, cách 4, hai bản `Solution`) trên 500 danh sách ngẫu nhiên nhỏ (`random.seed` cố định để chạy lại ra y hệt), với brute force làm trọng tài, và đếm xem Kahn và DFS trả **chuỗi khác nhau** (nhưng đều hợp lệ) bao nhiêu lần.

In [25]:
class SolutionDFS:
    def alienOrder(self, words: List[str]) -> str:
        adj = {c: [] for w in words for c in w}
        for w1, w2 in zip(words, words[1:]):
            for a, b in zip(w1, w2):
                if a != b:
                    if b not in adj[a]:
                        adj[a].append(b)
                    break
            else:
                if len(w1) > len(w2):
                    return ""

        WHITE, GRAY, BLACK = 0, 1, 2
        color = {c: WHITE for c in adj}
        post = []

        def dfs(u):
            color[u] = GRAY
            for v in adj[u]:
                if color[v] == GRAY or (color[v] == WHITE and not dfs(v)):
                    return False
            color[u] = BLACK
            post.append(u)
            return True

        for c in adj:
            if color[c] == WHITE and not dfs(c):
                return ""
        return "".join(reversed(post))


run_tests(SolutionDFS().alienOrder)

PASS Example 1                          -> 'wertf'
PASS Example 2                          -> 'zx'
PASS Example 3 (chu trình)              -> ''
PASS tiền tố đứng sau                   -> ''
PASS tiền tố đứng trước                 -> 'cba'
PASS một từ                             -> 'cba'
PASS từ trùng nhau                      -> 'z'
PASS chỉ so chữ khác đầu tiên           -> 'ab'
PASS nhiều mảnh rời                     -> 'cbda'
PASS chu trình + chữ tự do              -> ''
PASS cạnh trùng                         -> 'abc'
PASS chu trình 3 chữ                    -> ''
PASS 100 từ, bảng ẩn 26 chữ             -> 'abctjgkvqxfm...'
PASS như trên, chép từ đầu xuống cuối   -> ''


In [26]:
random.seed(2026)
solvers = {
    "cách 2 (mọi cặp)": alien_order_all_pairs,
    "cách 3 (Kahn)": alien_order_kahn,
    "cách 4 (DFS)": alien_order_dfs,
    "Solution": Solution().alienOrder,
    "SolutionDFS": SolutionDFS().alienOrder,
}
n_valid = n_invalid = n_diff = 0
for _ in range(500):
    words = random_words("abcdef", max_n=6, max_len=4)
    expect_valid = alien_order_brute(words) != ""            # trọng tài
    answers = {name: f(words) for name, f in solvers.items()}
    for name, ans in answers.items():
        assert is_valid_alien_answer(words, ans, expect_valid), (name, words, ans)
    if expect_valid:
        n_valid += 1
        n_diff += answers["cách 3 (Kahn)"] != answers["cách 4 (DFS)"]
    else:
        n_invalid += 1
print(f"500 danh sách ngẫu nhiên: {n_valid} hợp lệ, {n_invalid} không hợp lệ -> cả {len(solvers)} cách đều PASS")
print(f"Trong {n_valid} danh sách hợp lệ, Kahn và DFS trả chuỗi KHÁC nhau ở {n_diff} danh sách (nhưng đều hợp lệ)")

500 danh sách ngẫu nhiên: 376 hợp lệ, 124 không hợp lệ -> cả 5 cách đều PASS
Trong 376 danh sách hợp lệ, Kahn và DFS trả chuỗi KHÁC nhau ở 188 danh sách (nhưng đều hợp lệ)


## 14. Vì sao cách chính đúng? (invariant)

**Bước A — đồ thị nắm trọn thông tin của bài.** Với một bảng chữ cái σ bất kỳ, xét từng cặp liền kề `(w1, w2)`:

- kết cục **edge** `a → b`: cặp đúng thứ tự theo σ ⟺ σ đặt `a` trước `b` (Luật 1 — chỉ vị trí khác đầu tiên quyết định);
- kết cục **ok**: cặp luôn đúng thứ tự, với mọi σ (Luật 2);
- kết cục **invalid**: cặp luôn sai, với mọi σ (Luật 2).

Kết hợp với bổ đề "liền kề là đủ" (mục 7, nhờ tính bắc cầu): `words` sắp xếp theo σ ⟺ không có cặp invalid **và** σ thỏa mọi cạnh. Vậy nếu `build_graph` gặp invalid thì trả `""` là đúng; ngược lại **đáp án hợp lệ = topological order của đồ thị** (chứa đủ mọi chữ cái).

**Bước B — Kahn trả về topological order khi nó tồn tại.** Invariant: *tại mọi thời điểm, `indeg[v]` bằng số cạnh `u → v` mà `u` **chưa** được xếp; một chữ vào queue đúng một lần, ngay khi `indeg` của nó về 0.*

- **Không xếp sai**: khi `v` được xếp, `indeg[v] = 0`, nghĩa là mọi `u` có cạnh `u → v` đều đã được xếp trước `v`. Vậy mọi cạnh chỉ từ trái sang phải.
- **Không bỏ sót khi không có chu trình**: giả sử queue cạn mà vẫn còn chữ chưa xếp. Mỗi chữ còn lại có `indeg ≥ 1`, tức có ít nhất một "người chặn" **cũng chưa xếp**. Đi ngược từ một chữ còn lại sang người chặn của nó, rồi người chặn của người chặn, ... Vì chỉ có hữu hạn chữ, sớm muộn phải **gặp lại** một chữ đã đi qua → đó là một chu trình. Vậy: **không có chu trình ⇒ Kahn xếp đủ mọi chữ**.
- **Có chu trình ⇒ trả `""`**: các chữ trên chu trình luôn còn một người chặn trên chính chu trình đó, nên không bao giờ về 0 → `len(order) < k` → trả `""`. Và đúng là không có đáp án (mục 6: chu trình bắt một chữ đứng trước chính nó).

**Bước C — DFS 3 màu.** Invariant: *các đỉnh GRAY luôn là đúng đường đi từ gốc DFS hiện tại tới đỉnh đang xét (chính là call stack).*

- Gặp cạnh `u → v` với `v` GRAY ⇒ có đường `v → ... → u → v` ⇒ chu trình thật, trả `""` là đúng.
- Nếu đồ thị có chu trình, DFS **chắc chắn** gặp cạnh tới GRAY: gọi `v` là đỉnh của chu trình được tô GRAY **đầu tiên**; lúc đó mọi đỉnh khác của chu trình còn WHITE và đi tới được từ `v` qua các đỉnh WHITE, nên DFS từ `v` sẽ thăm tới đỉnh `u` đứng ngay trước `v` trên chu trình trước khi `v` xong; tại `u`, cạnh `u → v` gặp `v` đang GRAY.
- Không có chu trình thì mọi cạnh `u → v` có `v` vào `post` trước `u` (mục 12) ⇒ đảo ngược `post` là topological order.

## 15. Độ phức tạp & bảng so sánh các cách

Nhắc ký hiệu: `n` = số từ, `L` = độ dài từ dài nhất, `C` = tổng số ký tự (≤ 10^4), `k` = số chữ cái phân biệt (≤ 26), `E` = số cạnh phân biệt (≤ `n - 1` vì mỗi cặp liền kề cho tối đa 1 cạnh, và ≤ `k(k-1)`).

**Cách 3 (Kahn) chi tiết:**

- Tạo đỉnh: duyệt mọi ký tự → `O(C)`.
- Rút cạnh: mỗi cặp liền kề so tối đa `min(len(w1), len(w2))` ký tự → tổng `O(C)`; mỗi cặp thêm một phép `b not in adj[a]` tốn `O(k)` → `O(n · k)` (nếu dùng `set` thì `O(1)` mỗi phép).
- Kahn: mỗi đỉnh vào/ra queue đúng 1 lần → `O(k)`; mỗi cạnh được duyệt đúng 1 lần → `O(E)`.
- **Tổng thời gian** `O(C + n·k + k + E)`; vì `k ≤ 26` là hằng số nhỏ, thực chất là `O(C)` — tuyến tính theo kích thước input (không thể tốt hơn đáng kể, vì ít nhất phải đọc input).
- **Bộ nhớ phụ** `O(k + E)` cho `adj`, `indeg`, `queue`, `order` — tối đa vài trăm phần tử.

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | Thử mọi hoán vị, kiểm tra bằng trọng tài | `O(k! · C)` | `O(k + C)` | Chỉ dùng khi `k ≤ 8`; làm trọng tài |
| 2. Mọi cặp + quét chữ tự do | So `n(n-1)/2` cặp; mỗi vòng quét tìm chữ không bị chặn | `O(n² · L + k³)` | `O(k²)` | Đúng, qua được constraints nhỏ nhưng lãng phí |
| 3. **Kahn** (BFS in-degree) | Cặp liền kề + hàng đợi chữ tự do | `O(C + n·k)` ≈ `O(C)` | `O(k + E)` | **Cách chính**: không đệ quy, dễ trace, dễ biến thể |
| 4. DFS 3 màu | Cặp liền kề + post-order đảo ngược | `O(C + n·k)` ≈ `O(C)` | `O(k + E)` + stack đệ quy `O(k)` | Gọn; GRAY phát hiện chu trình |

Space của brute force gồm `O(C)` cho list thứ hạng `keys` của trọng tài; cách 2 lưu tối đa `k(k-1)` luật phân biệt nên `O(k²)`.

## 16. Lỗi thường gặp

### Nhóm 1 — Rút cạnh sai từ một cặp từ

**Lỗi 1.1: Quên trường hợp tiền tố đứng sau.**

```python
for a, b in zip(w1, w2):
    if a != b:
        add_edge(a, b)
        break
# (không có else) -> với ["abc", "ab"] không sinh cạnh nào, coi như "ổn"
```

Vì sao sai: "không có cạnh" **không** có nghĩa là hợp lệ. `"ab"` là tiền tố thật sự của `"abc"` nên phải đứng trước trong **mọi** bảng chữ cái. Bản lỗi trả về một hoán vị của `abc` thay vì `""`.
**Sửa:** thêm `else: if len(w1) > len(w2): return ""` (hoặc dùng biến cờ `found` như code demo bên dưới).

**Lỗi 1.2: Không `break` sau chữ khác nhau đầu tiên.**

```python
for a, b in zip(w1, w2):
    if a != b:
        add_edge(a, b)      # thiếu break -> tiếp tục rút "luật" từ các chữ phía sau
```

Vì sao sai: theo Luật 1, chỉ chữ khác nhau **đầu tiên** có ý nghĩa. Với `["ab", "ba"]`, bản lỗi rút cả `a → b` lẫn `b → a` → chu trình giả → trả `""`, trong khi `"ab"` là đáp án đúng.
**Sửa:** `break` ngay sau khi xử lý chữ khác nhau đầu tiên.

In [27]:
def graph_bug(words, prefix_check=True, use_break=True):
    # build_graph có thể "tắt" kiểm tra tiền tố hoặc "quên" break để thấy lỗi
    adj = {c: [] for w in words for c in w}
    indeg = {c: 0 for c in adj}
    for w1, w2 in zip(words, words[1:]):
        found = False                      # biến cờ: thay cho for ... else
        for a, b in zip(w1, w2):
            if a != b:
                found = True
                if b not in adj[a]:
                    adj[a].append(b)
                    indeg[b] += 1
                if use_break:
                    break
        if prefix_check and not found and len(w1) > len(w2):
            return None
    return adj, indeg

def solve_with(graph):
    # phần Kahn + kiểm tra độ dài, dùng chung cho các bản lỗi
    if graph is None:
        return ""
    adj, indeg = graph
    order = kahn_order(adj, indeg)
    return "".join(order) if len(order) == len(adj) else ""

w = ["abc", "ab"]
print("Lỗi 1.1:", w, "| bản lỗi ->", repr(solve_with(graph_bug(w, prefix_check=False))),
      "| bản đúng ->", repr(alien_order_kahn(w)))
w = ["ab", "ba"]
print("Lỗi 1.2:", w, "| bản lỗi ->", repr(solve_with(graph_bug(w, use_break=False))),
      "| bản đúng ->", repr(alien_order_kahn(w)))

Lỗi 1.1: ['abc', 'ab'] | bản lỗi -> 'abc' | bản đúng -> ''
Lỗi 1.2: ['ab', 'ba'] | bản lỗi -> '' | bản đúng -> 'ab'


### Nhóm 2 — Dựng đồ thị thiếu chữ, hoặc đếm in-degree sai

**Lỗi 2.1: Chỉ tạo đỉnh cho các chữ nằm trong một cạnh nào đó.**

```python
# chỉ khi rút được a -> b mới tạo đỉnh:
if a not in adj: adj[a] = []; indeg[a] = 0
if b not in adj: adj[b] = []; indeg[b] = 0
```

Vì sao sai: đề yêu cầu trả **mọi** chữ xuất hiện. Một chữ không dính luật nào (như `a`, `c` trong `["ab", "adc"]`) vẫn là một chữ của bảng chữ cái, nhưng bản lỗi không bao giờ tạo đỉnh cho nó → đáp án thiếu chữ (`"bd"`). Kiểm tra `len(order) == len(adj)` cũng không cứu được, vì `adj` cũng thiếu y như vậy.
**Sửa:** tạo đỉnh cho mọi chữ của mọi từ **trước** khi rút cạnh.

**Lỗi 2.2: Chống trùng ở `adj` (dùng `set`) nhưng vẫn tăng `indeg` mỗi lần gặp lại cạnh.**

```python
adj[a].add(b)        # set: cạnh trùng chỉ lưu 1 lần
indeg[b] += 1        # ...nhưng in-degree tăng mỗi lần cạnh xuất hiện
```

Với `["ab", "ac", "bb", "bc"]`, cạnh `b → c` xuất hiện 2 lần (từ `ab/ac` và `bb/bc`). `indeg[c] = 2`, nhưng khi xếp `b` chỉ giảm được 1 lần (vì `adj[b]` chỉ chứa `c` một lần) → `c` kẹt ở 1 mãi → bản lỗi tưởng có chu trình → `""`. Đáp án đúng là `"abc"`.
**Sửa:** chỉ tăng `indeg[b]` khi cạnh **thật sự mới** (`if b not in adj[a]: ...`). Cách khác cũng đúng là **không** chống trùng ở cả hai phía (list `adj[a]` chứa `b` hai lần, `indeg[b]` tăng hai lần, khi xếp `a` sẽ giảm đủ hai lần). Điều quan trọng là **nhất quán**.

In [28]:
def graph_bug_missing_letters(words):
    adj, indeg = {}, {}
    for w1, w2 in zip(words, words[1:]):
        for a, b in zip(w1, w2):
            if a != b:
                for c in (a, b):                     # LỖI: chỉ tạo đỉnh cho chữ thuộc cạnh
                    if c not in adj:
                        adj[c] = []
                        indeg[c] = 0
                if b not in adj[a]:
                    adj[a].append(b)
                    indeg[b] += 1
                break
        else:
            if len(w1) > len(w2):
                return None
    return adj, indeg

def graph_bug_double_count(words):
    adj = {c: set() for w in words for c in w}       # set chống trùng cạnh...
    indeg = {c: 0 for c in adj}
    for w1, w2 in zip(words, words[1:]):
        for a, b in zip(w1, w2):
            if a != b:
                adj[a].add(b)
                indeg[b] += 1                        # LỖI: ...nhưng in-degree đếm cả cạnh trùng
                break
        else:
            if len(w1) > len(w2):
                return None
    return adj, indeg

w = ["ab", "adc"]
print("Lỗi 2.1:", w, "| bản lỗi ->", repr(solve_with(graph_bug_missing_letters(w))),
      "| bản đúng ->", repr(alien_order_kahn(w)))
w = ["ab", "ac", "bb", "bc"]
print("Lỗi 2.2:", w, "| bản lỗi ->", repr(solve_with(graph_bug_double_count(w))),
      "| bản đúng ->", repr(alien_order_kahn(w)))
print("         indeg của bản lỗi:", graph_bug_double_count(w)[1], "| bản đúng:", build_graph(w)[1])

Lỗi 2.1: ['ab', 'adc'] | bản lỗi -> 'bd' | bản đúng -> 'abcd'
Lỗi 2.2: ['ab', 'ac', 'bb', 'bc'] | bản lỗi -> '' | bản đúng -> 'abc'
         indeg của bản lỗi: {'a': 0, 'b': 1, 'c': 2} | bản đúng: {'a': 0, 'b': 1, 'c': 1}


### Nhóm 3 — Quên kiểm tra chu trình ở cuối Kahn

```python
...
return "".join(order)          # thiếu: nếu len(order) < số chữ cái thì return ""
```

Vì sao sai: khi có chu trình, Kahn **không báo lỗi gì** — hàng đợi chỉ lặng lẽ cạn sớm. Nguy hiểm hơn: với Example 3 `["z","x","z"]` bản lỗi "may mắn" vẫn ra `""` (vì không chữ nào có in-degree 0 nên `order` rỗng), nên test Example 3 vẫn qua và bạn tưởng code đúng! Nhưng với `["a","b","c","b"]` (chu trình `b ⇄ c` cộng một chữ tự do `a`), bản lỗi trả `"a"` — sai.

**Sửa:** luôn so `len(order)` với số chữ cái. **Bài học test:** Example 3 **không đủ** để kiểm tra phần phát hiện chu trình — cần ca có chu trình **kèm chữ tự do**.

In [29]:
def alien_order_bug_no_cycle_check(words):
    graph = build_graph(words)
    if graph is None:
        return ""
    adj, indeg = graph
    return "".join(kahn_order(adj, indeg))       # LỖI: quên so độ dài

for w in [ex3, ["a", "b", "c", "b"]]:
    print(f"{str(w):22} bản lỗi -> {alien_order_bug_no_cycle_check(w)!r:5} | bản đúng -> {alien_order_kahn(w)!r}")

['z', 'x', 'z']        bản lỗi -> ''    | bản đúng -> ''
['a', 'b', 'c', 'b']   bản lỗi -> 'a'   | bản đúng -> ''


### Nhóm 4 — DFS chỉ dùng 2 trạng thái (visited / chưa visited)

**Lỗi 4.1: Gặp đỉnh đã visited thì bỏ qua** → không bao giờ phát hiện chu trình. Với `["z","x","z"]`: `dfs(z)` → `dfs(x)` → `x` nhìn sang `z`: "đã visited, bỏ qua" → kết quả `"zx"` thay vì `""`.

**Lỗi 4.2: Gặp đỉnh đã visited thì báo chu trình** → báo động giả khi gặp một đỉnh **đã xong** từ nhánh khác. Với `["a","ba","bc","c"]` (cạnh `a → b`, `a → c`, `b → c` — một "tam giác" **không** có chu trình): `dfs(a)` → `dfs(b)` → `dfs(c)` xong; quay về `a`, nhìn sang `c` "đã visited" → báo chu trình → `""`, trong khi đáp án đúng là `"abc"`.

**Sửa:** dùng 3 màu. Chỉ GRAY (đang trên call stack) mới là chu trình; BLACK (đã xong) là an toàn.

In [30]:
def alien_order_dfs_2state(words, visited_means_cycle):
    graph = build_graph(words)
    if graph is None:
        return ""
    adj, _ = graph
    visited, post = set(), []

    def dfs(u):
        visited.add(u)
        for v in adj[u]:
            if v in visited:
                if visited_means_cycle:
                    return False        # Lỗi 4.2: không phân biệt "đang thăm" với "đã xong"
                continue                # Lỗi 4.1: bỏ qua luôn cả trường hợp chu trình
            if not dfs(v):
                return False
        post.append(u)
        return True

    for c in adj:
        if c not in visited and not dfs(c):
            return ""
    return "".join(reversed(post))

w = ex3
print("Lỗi 4.1:", w, "| bản lỗi ->", repr(alien_order_dfs_2state(w, visited_means_cycle=False)),
      "| 3 màu ->", repr(alien_order_dfs(w)))
w = ["a", "ba", "bc", "c"]
print("Lỗi 4.2:", w, "| bản lỗi ->", repr(alien_order_dfs_2state(w, visited_means_cycle=True)),
      "| 3 màu ->", repr(alien_order_dfs(w)))

Lỗi 4.1: ['z', 'x', 'z'] | bản lỗi -> 'zx' | 3 màu -> ''
Lỗi 4.2: ['a', 'ba', 'bc', 'c'] | bản lỗi -> '' | 3 màu -> 'abc'


## 17. Mẹo nhận dạng pattern

- Đề cho các **ràng buộc thứ tự từng cặp** "A phải trước B" (môn học tiên quyết, công việc phụ thuộc nhau, gói phần mềm cần cài trước, công thức nấu ăn cần nguyên liệu...) và hỏi **một thứ tự tổng thể** hoặc "có làm được không" → nghĩ ngay tới **topological sort**.
- "Có làm được không?" chính là "đồ thị có chu trình không?" → Kahn (so `len(order)` với số đỉnh) hoặc DFS 3 màu (gặp GRAY).
- Dữ liệu là **một danh sách đã sắp xếp theo luật lạ** → so sánh **phần tử liền kề**, mỗi cặp cho tối đa **một** ràng buộc (vị trí khác nhau đầu tiên). Mẹo "liền kề là đủ" đúng nhờ **tính bắc cầu**.
- Có trường hợp "không hợp lệ" **không** biểu diễn được bằng cạnh không? Ở bài này là **tiền tố đứng sau** — phải kiểm tra riêng. Hãy luôn tự hỏi câu này.
- Luôn khởi tạo **mọi** đỉnh (kể cả đỉnh cô lập) trước khi thêm cạnh.
- Kahn hay DFS? Kahn dễ trace, không đệ quy, dễ biến thể: đổi `deque` thành heap để lấy thứ tự nhỏ nhất; kiểm tra `len(queue) > 1` để biết đáp án có duy nhất không. DFS ngắn gọn khi đã quen đệ quy, và 3 màu là công cụ chuẩn để **phát hiện chu trình** trên đồ thị có hướng.
- Ba câu hỏi trước khi code: (1) Đỉnh là gì? (2) Cạnh `u → v` nghĩa là gì, lấy từ đâu ra? (3) Khi nào trả "không hợp lệ"?

## 18. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| #953 Verifying an Alien Dictionary (Easy) | Bài "ngược": cho bảng chữ cái, kiểm tra danh sách — chính là `is_sorted_by_order` |
| #207 Course Schedule (Medium) | Chỉ hỏi "học hết được không?" = đồ thị có chu trình không |
| #210 Course Schedule II (Medium) | Trả về một thứ tự học — Kahn y hệt (Bài tập 5) |
| #444 Sequence Reconstruction (Medium, Premium) | Hỏi topological order có **duy nhất** không (giống Bài tập 6) |
| #802 Find Eventual Safe States (Medium) | DFS 3 màu: đỉnh nào không dẫn vào chu trình |
| #2115 Find All Possible Recipes from Given Supplies (Medium) | Kahn với đỉnh là chuỗi (tên món, nguyên liệu) |
| #1203 Sort Items by Groups Respecting Dependencies (Hard) | Topological sort hai tầng (nhóm, rồi phần tử) |
| #329 Longest Increasing Path in a Matrix (Hard) | Thứ tự tô-pô / DFS + memo trên lưới |

**Biến thể "thứ tự nhỏ nhất"**: nếu đề yêu cầu, trong các đáp án hợp lệ, trả về chuỗi nhỏ nhất theo bảng chữ cái **thông thường** (theo mình biết, bản Alien Dictionary trên LintCode — bài 892 — có yêu cầu này), chỉ cần thay `deque` bằng **heap** (hàng đợi ưu tiên): mỗi lần lấy ra chữ **nhỏ nhất** trong các chữ đang tự do. Lấy chữ nhỏ nhất ở mỗi bước là tối ưu vì vị trí đầu tiên khác nhau quyết định thứ tự chuỗi — giống hệt Luật 1.

Module `heapq` (thư viện chuẩn) dùng một list làm **min-heap**: `heapq.heapify(h)` biến list `h` thành heap, `heapq.heappush(h, x)` thêm `x`, `heapq.heappop(h)` lấy ra phần tử **nhỏ nhất** hiện có; push/pop đều `O(log size)`.

In [31]:
import heapq

h = []
for x in ["x", "a", "y"]:
    heapq.heappush(h, x)
print("heappop lần lượt:", [heapq.heappop(h) for _ in range(3)])   # luôn ra nhỏ nhất trước

def alien_order_smallest(words):
    graph = build_graph(words)
    if graph is None:
        return ""
    adj, indeg = graph
    heap = [c for c in adj if indeg[c] == 0]
    heapq.heapify(heap)                    # biến list thành heap
    order = []
    while heap:
        u = heapq.heappop(heap)            # chữ tự do NHỎ NHẤT
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                heapq.heappush(heap, v)
    return "".join(order) if len(order) == len(adj) else ""

words = ["xa", "xb", "ya", "yc", "c", "z"]
print("Kahn + deque :", alien_order_kahn(words))
print("Kahn + heap  :", alien_order_smallest(words))
all_valid = sorted("".join(p) for p in permutations(unique_letters(words)) if is_sorted_by_order(words, p))
print("Nhỏ nhất theo brute force:", all_valid[0], "| tổng số đáp án hợp lệ:", len(all_valid))

heappop lần lượt: ['a', 'x', 'y']
Kahn + deque : xaybcz
Kahn + heap  : abxycz
Nhỏ nhất theo brute force: abxycz | tổng số đáp án hợp lệ: 12


## 19. Bài tập vận dụng

Làm theo thứ tự; đừng xem đáp án (mục 21) trước khi tự thử. Gợi ý theo 3 mức ở mục 20.

### Bài 1 — Tính tay (không code)

`words = ["ba", "bc", "ac", "cab"]`. Hãy:

1. Lập bảng các cặp liền kề → vị trí khác đầu tiên → luật.
2. Vẽ đồ thị, tính in-degree từng chữ (chữ theo thứ tự xuất hiện).
3. Chạy Kahn bằng tay, ghi `queue` và `order` sau mỗi bước.
4. Đáp án là gì? Có duy nhất không? Vì sao?

### Bài 2 — Đoán output: hợp lệ hay `""`?

Không chạy code, đoán mỗi danh sách cho đáp án `""` hay một chuỗi hợp lệ, và giải thích bằng **một câu** (tiền tố? chu trình? hay hợp lệ — khi đó nêu một đáp án):

| # | `words` |
|---|---|
| a | `["abc", "ab"]` |
| b | `["ab", "abc"]` |
| c | `["a", "b", "a"]` |
| d | `["z", "z"]` |
| e | `["ca", "cb", "ba"]` |
| f | `["ac", "ab", "b", "c"]` |

### Bài 3 — Điền TODO: tự dựng đồ thị

Viết `exercise_build_graph(words)` trả về `(adj, indeg)` giống `build_graph`, hoặc `None` nếu gặp tiền tố không hợp lệ. Yêu cầu: tạo đỉnh cho **mọi** chữ, chỉ so cặp liền kề, `break` sau chữ khác đầu tiên, **không** đếm trùng in-degree. Không gọi lại `build_graph` hay `compare_pair`.

### Bài 4 — Tự viết lại cách chính

Viết `exercise_alien_order(words)` từ đầu (dựng đồ thị + Kahn với `deque` + kiểm tra chu trình), cố gắng **không nhìn lên** code ở trên. Test bằng `is_valid_alien_answer`, **không** so với chuỗi cố định.

In [32]:
def exercise_build_graph(words):
    # TODO 1: tạo adj[ch] = [] và indeg[ch] = 0 cho MỌI chữ của MỌI từ
    # TODO 2: duyệt các cặp liền kề (w1, w2) bằng zip(words, words[1:])
    # TODO 3: tìm chữ khác nhau đầu tiên a != b:
    #         nếu cạnh a -> b còn mới: thêm b vào adj[a], tăng indeg[b]; rồi break
    # TODO 4: nếu không có chữ khác nhau và len(w1) > len(w2): return None
    # TODO 5: return adj, indeg
    pass

# Tự làm xong rồi mới mở test:
# adj, indeg = exercise_build_graph(["wrt", "wrf", "er", "ett", "rftt"])
# assert adj == {"w": ["e"], "r": ["t"], "t": ["f"], "f": [], "e": ["r"]}
# assert indeg == {"w": 0, "r": 1, "t": 1, "f": 1, "e": 1}
# assert exercise_build_graph(["abc", "ab"]) is None
# assert exercise_build_graph(["ab", "ac", "bb", "bc"])[1] == {"a": 0, "b": 1, "c": 1}
# assert exercise_build_graph(["ab", "adc"])[0] == {"a": [], "b": ["d"], "d": [], "c": []}

In [33]:
def exercise_alien_order(words: List[str]) -> str:
    # TODO 1: dựng đồ thị (giống Bài 3); gặp tiền tố không hợp lệ -> return ""
    # TODO 2: queue = deque(các chữ có indeg 0)
    # TODO 3: lặp: popleft, thêm vào order, giảm indeg các chữ kề, chữ nào về 0 thì append
    # TODO 4: nếu len(order) < số chữ cái -> return "" (chu trình); ngược lại "".join(order)
    pass

# Gợi ý test (dùng hàm kiểm tra hợp lệ, KHÔNG so chuỗi cố định):
# for w in [ex1, ex2, ex3, ["abc", "ab"], ["ab", "adc"], ["z", "z"], ["a", "b", "c", "b"], ["ab", "ac", "bb", "bc"]]:
#     assert is_valid_alien_answer(w, exercise_alien_order(w)), w

### Bài 5 — Biến thể: #210 Course Schedule II

Có `numCourses` môn học đánh số `0 .. numCourses - 1`. `prerequisites[i] = [a, b]` nghĩa là **muốn học `a` thì phải học `b` trước**. Trả về **một** thứ tự học hết các môn; nếu không thể (có vòng tiên quyết), trả về `[]`. Nhiều đáp án thì trả bất kỳ.

```text
numCourses = 2, prerequisites = [[1,0]]                    -> [0,1]
numCourses = 4, prerequisites = [[1,0],[2,0],[3,1],[3,2]]  -> [0,2,1,3] hoặc [0,1,2,3]
numCourses = 1, prerequisites = []                         -> [0]
```

Chú ý chiều cạnh! Cell có sẵn hàm kiểm tra `is_valid_course_order` (không phải TODO) để bạn test.

### Bài 6 — Mini challenge: bảng chữ cái có **duy nhất** không?

Viết `exercise_is_unique_order(words)` trả `True` khi và chỉ khi có **đúng một** bảng chữ cái hợp lệ (danh sách không hợp lệ → 0 bảng → `False`). Yêu cầu dùng Kahn, **không** dùng brute force; nhưng hãy kiểm tra kết quả với `count_valid_orders(words) == 1` trên input nhỏ. Ví dụ: Example 1 → `True`; `["ab", "adc"]` → `False`; `["z"]` → `True`.

In [34]:
def is_valid_course_order(numCourses, prerequisites, order, expect_possible=True):
    # Hàm kiểm tra (không phải TODO): order có học đủ mọi môn và đúng tiên quyết không
    if not expect_possible:
        return order == []
    if sorted(order) != list(range(numCourses)):
        return False
    pos = {c: i for i, c in enumerate(order)}
    return all(pos[b] < pos[a] for a, b in prerequisites)

def exercise_find_order(numCourses: int, prerequisites: List[List[int]]) -> List[int]:
    # TODO 1: đỉnh = môn 0..numCourses-1; [a, b] nghĩa là cạnh b -> a (b học trước a)
    # TODO 2: tính indeg, cho các môn indeg 0 vào deque
    # TODO 3: Kahn như bài chính
    # TODO 4: đủ numCourses môn thì trả order, ngược lại trả []
    pass

# Gợi ý test:
# assert is_valid_course_order(2, [[1, 0]], exercise_find_order(2, [[1, 0]]))
# assert is_valid_course_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]], exercise_find_order(4, [[1, 0], [2, 0], [3, 1], [3, 2]]))
# assert is_valid_course_order(1, [], exercise_find_order(1, []))
# assert exercise_find_order(2, [[1, 0], [0, 1]]) == []

In [35]:
def exercise_is_unique_order(words: List[str]) -> bool:
    # TODO 1: dựng đồ thị; tiền tố không hợp lệ -> False
    # TODO 2: chạy Kahn; nếu có lúc queue chứa >= 2 chữ thì ... ?
    # TODO 3: cuối cùng nhớ kiểm tra không có chu trình
    pass

# Gợi ý test:
# assert exercise_is_unique_order(ex1) is True
# assert exercise_is_unique_order(["ab", "adc"]) is False
# assert exercise_is_unique_order(ex3) is False
# assert exercise_is_unique_order(["z"]) is True
# random.seed(1)
# for _ in range(300):
#     w = random_words()
#     assert exercise_is_unique_order(w) == (count_valid_orders(w) == 1), w

## 20. Gợi ý theo mức

**Bài 1**
- Mức 1: chỉ so 3 cặp liền kề `ba/bc`, `bc/ac`, `ac/cab`.
- Mức 2: có một luật xuất hiện **hai lần** — chỉ tính một cạnh.
- Mức 3: các luật là `a → c` và `b → a`; in-degree `b:0, a:1, c:1`.

**Bài 2**
- Mức 1: chỉ có hai nguồn gốc của `""`: tiền tố đứng sau, và chu trình.
- Mức 2: với mỗi cặp liền kề, xếp vào một trong 3 kết cục edge / ok / invalid; rồi xem các cạnh có tạo vòng không.
- Mức 3: hai danh sách có chu trình, một danh sách có tiền tố đứng sau.

**Bài 3**
- Mức 1: hai giai đoạn tách bạch: (1) tạo đỉnh cho mọi chữ, (2) rút cạnh từ cặp liền kề.
- Mức 2: dùng `for ... else` (hoặc biến cờ `found`) để biết "không tìm thấy chữ khác nhau".
- Mức 3: `if a != b: if b not in adj[a]: adj[a].append(b); indeg[b] += 1` rồi `break`; khối `else: if len(w1) > len(w2): return None`.

**Bài 4**
- Mức 1: ba mảnh: dựng đồ thị → Kahn → kiểm tra độ dài.
- Mức 2: `queue = deque(c for c in adj if indeg[c] == 0)`; vòng `while queue:`.
- Mức 3: `return "".join(order) if len(order) == len(adj) else ""`.

**Bài 5**
- Mức 1: đỉnh = môn học; cạnh `u → v` = "học `u` trước `v`".
- Mức 2: `[a, b]` nghĩa là cạnh `b → a` — đừng đảo chiều! Có thể dùng list `adj = [[] for _ in range(numCourses)]` vì đỉnh là số `0..n-1`.
- Mức 3: trả `order` nếu `len(order) == numCourses`, ngược lại `[]`.

**Bài 6**
- Mức 1: khi nào ta có **lựa chọn**? Khi có từ 2 chữ trở lên cùng tự do một lúc.
- Mức 2: trong Kahn, đó chính là lúc `len(queue) >= 2`: hai chữ `u`, `v` cùng tự do thì xếp `u` trước hay `v` trước đều hoàn thành được (nếu không có chu trình) → ít nhất 2 đáp án.
- Mức 3: đầu mỗi vòng `while`: `if len(queue) > 1: return False`; cuối cùng `return count == len(adj)` (đủ chữ = không có chu trình).

## 21. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

| Cặp liền kề | Vị trí khác đầu tiên | Luật |
|---|---|---|
| `ba` / `bc` | 1: `a` vs `c` | `a → c` |
| `bc` / `ac` | 0: `b` vs `a` | `b → a` |
| `ac` / `cab` | 0: `a` vs `c` | `a → c` (trùng, bỏ qua) |

Đồ thị `b → a → c`; chữ theo thứ tự xuất hiện `b a c`, in-degree `b:0, a:1, c:1`.

| Bước | Lấy ra | indeg thay đổi | queue sau bước | order |
|---|---|---|---|---|
| 0 | — | — | `[b]` | `""` |
| 1 | `b` | `a: 1→0` | `[a]` | `b` |
| 2 | `a` | `c: 1→0` | `[c]` | `ba` |
| 3 | `c` | — | `[]` | `bac` |

Đáp án `"bac"` và là **duy nhất**, vì hàng đợi lúc nào cũng chỉ có đúng 1 chữ (không có lựa chọn nào). Vì duy nhất nên ở đây so chuỗi cố định cũng an toàn.

### Đáp án Bài 2

| # | `words` | Kết quả | Lý do |
|---|---|---|---|
| a | `["abc", "ab"]` | `""` | `"ab"` là tiền tố thật sự của `"abc"` nhưng đứng sau |
| b | `["ab", "abc"]` | hợp lệ, ví dụ `"abc"` | tiền tố đứng trước là đúng luật; không có luật nào → mọi hoán vị của `a, b, c` (6 đáp án) |
| c | `["a", "b", "a"]` | `""` | `a → b` và `b → a`: chu trình |
| d | `["z", "z"]` | `"z"` | hai từ bằng nhau được phép đứng cạnh nhau |
| e | `["ca", "cb", "ba"]` | hợp lệ, ví dụ `"cab"` | luật `a → b`, `c → b`; `a`, `c` thứ tự tùy ý (2 đáp án) |
| f | `["ac", "ab", "b", "c"]` | `""` | luật `c → b`, `a → b`, `b → c`: chu trình `b ⇄ c` |

In [36]:
# Đáp án Bài 1
w1_ex = ["ba", "bc", "ac", "cab"]
print("Bài 1:", Solution().alienOrder(w1_ex), "| số đáp án hợp lệ:", count_valid_orders(w1_ex))
assert Solution().alienOrder(w1_ex) == "bac" and count_valid_orders(w1_ex) == 1

# Đáp án Bài 2
bai2 = {
    "a": (["abc", "ab"], False), "b": (["ab", "abc"], True), "c": (["a", "b", "a"], False),
    "d": (["z", "z"], True), "e": (["ca", "cb", "ba"], True), "f": (["ac", "ab", "b", "c"], False),
}
for key, (w, expect_valid) in bai2.items():
    ans = Solution().alienOrder(w)
    assert (ans != "") == expect_valid and is_valid_alien_answer(w, ans), (key, w, ans)
    print(f"Bài 2{key}: {str(w):22} -> {ans!r:6} (số đáp án hợp lệ: {count_valid_orders(w)})")

# Đáp án Bài 3
def answer_build_graph(words):
    adj = {c: [] for w in words for c in w}
    indeg = {c: 0 for c in adj}
    for w1, w2 in zip(words, words[1:]):
        for a, b in zip(w1, w2):
            if a != b:
                if b not in adj[a]:
                    adj[a].append(b)
                    indeg[b] += 1
                break
        else:
            if len(w1) > len(w2):
                return None
    return adj, indeg

adj, indeg = answer_build_graph(["wrt", "wrf", "er", "ett", "rftt"])
assert adj == {"w": ["e"], "r": ["t"], "t": ["f"], "f": [], "e": ["r"]}
assert indeg == {"w": 0, "r": 1, "t": 1, "f": 1, "e": 1}
assert answer_build_graph(["abc", "ab"]) is None
assert answer_build_graph(["ab", "ac", "bb", "bc"])[1] == {"a": 0, "b": 1, "c": 1}
assert answer_build_graph(["ab", "adc"])[0] == {"a": [], "b": ["d"], "d": [], "c": []}
print("Bài 3: PASS")

Bài 1: bac | số đáp án hợp lệ: 1
Bài 2a: ['abc', 'ab']          -> ''     (số đáp án hợp lệ: 0)
Bài 2b: ['ab', 'abc']          -> 'abc'  (số đáp án hợp lệ: 6)
Bài 2c: ['a', 'b', 'a']        -> ''     (số đáp án hợp lệ: 0)
Bài 2d: ['z', 'z']             -> 'z'    (số đáp án hợp lệ: 1)
Bài 2e: ['ca', 'cb', 'ba']     -> 'cab'  (số đáp án hợp lệ: 2)
Bài 2f: ['ac', 'ab', 'b', 'c'] -> ''     (số đáp án hợp lệ: 0)
Bài 3: PASS


In [37]:
# Đáp án Bài 4
def answer_alien_order(words: List[str]) -> str:
    graph = answer_build_graph(words)
    if graph is None:
        return ""
    adj, indeg = graph
    queue = deque(c for c in adj if indeg[c] == 0)
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return "".join(order) if len(order) == len(adj) else ""

for w in [ex1, ex2, ex3, ["abc", "ab"], ["ab", "adc"], ["z", "z"], ["a", "b", "c", "b"], ["ab", "ac", "bb", "bc"]]:
    assert is_valid_alien_answer(w, answer_alien_order(w)), w
random.seed(4)
for _ in range(300):
    w = random_words()
    assert is_valid_alien_answer(w, answer_alien_order(w)), w
print("Bài 4: PASS (ví dụ cố định + 300 bộ ngẫu nhiên)")

Bài 4: PASS (ví dụ cố định + 300 bộ ngẫu nhiên)


In [38]:
# Đáp án Bài 5 (#210 Course Schedule II)
def answer_find_order(numCourses: int, prerequisites: List[List[int]]) -> List[int]:
    adj = [[] for _ in range(numCourses)]
    indeg = [0] * numCourses
    for a, b in prerequisites:
        adj[b].append(a)              # b phải học trước a: cạnh b -> a
        indeg[a] += 1
    queue = deque(i for i in range(numCourses) if indeg[i] == 0)
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return order if len(order) == numCourses else []

for n_c, pre, possible in [(2, [[1, 0]], True), (4, [[1, 0], [2, 0], [3, 1], [3, 2]], True),
                           (1, [], True), (2, [[1, 0], [0, 1]], False), (3, [[1, 0], [2, 1], [0, 2]], False)]:
    res = answer_find_order(n_c, pre)
    assert is_valid_course_order(n_c, pre, res, possible), (n_c, pre, res)
    print(f"Bài 5: numCourses={n_c}, prerequisites={pre} -> {res}")

# Đáp án Bài 6
def answer_is_unique_order(words: List[str]) -> bool:
    graph = answer_build_graph(words)
    if graph is None:
        return False                  # 0 đáp án
    adj, indeg = graph
    queue = deque(c for c in adj if indeg[c] == 0)
    count = 0
    while queue:
        if len(queue) > 1:
            return False              # >= 2 chữ cùng tự do -> đổi thứ tự được -> không duy nhất
        u = queue.popleft()
        count += 1
        for v in adj[u]:
            indeg[v] -= 1
            if indeg[v] == 0:
                queue.append(v)
    return count == len(adj)          # mỗi bước bị ép buộc VÀ không có chu trình

assert answer_is_unique_order(ex1) is True
assert answer_is_unique_order(["ab", "adc"]) is False
assert answer_is_unique_order(ex3) is False
assert answer_is_unique_order(["z"]) is True
random.seed(1)
for _ in range(300):
    w = random_words()
    assert answer_is_unique_order(w) == (count_valid_orders(w) == 1), w
print("Bài 6: PASS (khớp với brute force đếm số đáp án trên 300 bộ ngẫu nhiên)")

Bài 5: numCourses=2, prerequisites=[[1, 0]] -> [0, 1]
Bài 5: numCourses=4, prerequisites=[[1, 0], [2, 0], [3, 1], [3, 2]] -> [0, 1, 2, 3]
Bài 5: numCourses=1, prerequisites=[] -> [0]
Bài 5: numCourses=2, prerequisites=[[1, 0], [0, 1]] -> []
Bài 5: numCourses=3, prerequisites=[[1, 0], [2, 1], [0, 2]] -> []


Bài 6: PASS (khớp với brute force đếm số đáp án trên 300 bộ ngẫu nhiên)


## 22. Tổng kết: template nên nhớ + checklist trước khi submit

### Template 1 — Rút cạnh từ một danh sách đã sắp xếp theo luật lạ

```python
adj = {c: [] for w in words for c in w}          # MỌI chữ là một đỉnh
indeg = {c: 0 for c in adj}
for w1, w2 in zip(words, words[1:]):             # chỉ cặp LIỀN KỀ
    for a, b in zip(w1, w2):
        if a != b:                               # chữ khác nhau ĐẦU TIÊN
            if b not in adj[a]:
                adj[a].append(b); indeg[b] += 1
            break
    else:
        if len(w1) > len(w2):                    # tiền tố đứng sau
            return ""
```

### Template 2 — Kahn (dùng lại cho mọi bài topological sort)

```python
queue = deque(u for u in adj if indeg[u] == 0)
order = []
while queue:
    u = queue.popleft()
    order.append(u)
    for v in adj[u]:
        indeg[v] -= 1
        if indeg[v] == 0:
            queue.append(v)
ok = len(order) == len(adj)                      # False = có chu trình
```

**Câu thần chú**: *"Liền kề → chữ khác đầu tiên → một cạnh. Dài đứng trước tiền tố của nó → hỏng. In-degree 0 vào hàng. Xếp đủ là đúng, thiếu là có vòng."*

### Checklist trước khi submit

- [ ] Tạo đỉnh cho **mọi** chữ trong mọi từ (kể cả chữ không dính cạnh nào).
- [ ] Chỉ so sánh **cặp liền kề**; `break` ngay sau chữ khác nhau **đầu tiên**.
- [ ] Xử lý tiền tố: không có chữ khác nhau mà `len(w1) > len(w2)` → `""`.
- [ ] Không đếm trùng in-degree (chống trùng cạnh, hoặc nhất quán cả `adj` lẫn `indeg`).
- [ ] Cuối Kahn: `len(order) == số chữ cái`, nếu không → `""` (có chu trình).
- [ ] Nếu dùng DFS: 3 màu, chỉ GRAY mới là chu trình; nhớ **đảo ngược** post-order.
- [ ] Trả về **chuỗi** (`"".join(order)`), không phải list.
- [ ] Test bằng **hàm kiểm tra hợp lệ**, có đủ các ca: ví dụ đề, tiền tố đứng sau, từ trùng nhau, một từ, chu trình **kèm chữ tự do**, cạnh trùng, nhiều mảnh rời.